In [3]:

# Stage 3 — Step 3.5.1: Independent Injection Recalculation
# Goal: Independently recalculate Formula B from original
# source files and compare against saved final Parquet outputs.
#
# FIX: Normalize all date columns to datetime64[ns].
#
# STANDALONE NOTEBOOK:
# No previous notebook variables required.
#
# STRICT READ-ONLY:
# No file writing, modification, or export.

from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 80)
print("STAGE 3.5 — STEP 3.5.1")
print("INDEPENDENT INJECTION RECALCULATION")
print("=" * 80)


# ------------------------------------------------------------
# 1. Define source and output paths.
# ------------------------------------------------------------

ROOT_351 = Path(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration"
)

DE_351 = ROOT_351 / "Data_Engineering" / "Data_engineering"

DATA_351 = ROOT_351 / "Attack_Simulation" / "data"

SOURCE_PATHS_351 = {
    "Original User-Day": (
        DE_351 / "data" / "processed"
        / "user_day_r62_v1.parquet"
    ),
    "Original Baseline": (
        DE_351 / "data" / "features"
        / "user_baseline_r62_v1.csv"
    ),
    "Approved Scenario Catalog": (
        DATA_351 / "approved_scenario_catalog_r62.parquet"
    ),
    "Validated Schedule": (
        ROOT_351 / "Attack_Simulation"
        / "validation_inputs"
        / "validated_schedule_339.parquet"
    )
}

OUTPUT_PATHS_351 = {
    "GT Users": (
        DATA_351 / "ground_truth_users.parquet"
    ),
    "GT Channel-Days": (
        DATA_351 / "ground_truth_channel_days.parquet"
    ),
    "GT User-Days": (
        DATA_351 / "ground_truth_user_days.parquet"
    ),
    "Final Features": (
        DATA_351 / "behavior_features_injected_final.parquet"
    )
}

ALL_PATHS_351 = {
    **SOURCE_PATHS_351,
    **OUTPUT_PATHS_351
}

missing_351 = [
    (name, str(path))
    for name, path in ALL_PATHS_351.items()
    if not path.is_file()
]

if missing_351:
    for name, path in missing_351:
        print(f"MISSING — {name}: {path}")

    raise FileNotFoundError(
        "One or more required source files are missing."
    )

print("\nA. FILE DISCOVERY: PASS")


# ------------------------------------------------------------
# 2. Record independent SHA-256 fingerprints.
# ------------------------------------------------------------

def sha256_351(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(8 * 1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


fingerprints_351 = pd.DataFrame([
    {
        "Dataset": name,
        "Size_MB": round(
            path.stat().st_size / (1024 ** 2), 3
        ),
        "SHA256": sha256_351(path)
    }
    for name, path in ALL_PATHS_351.items()
])

display(fingerprints_351)


# ------------------------------------------------------------
# 3. Reload every original source and exported output.
# ------------------------------------------------------------

raw_351 = pd.read_parquet(
    SOURCE_PATHS_351["Original User-Day"],
    columns=[
        "user",
        "day",
        "is_weekend",
        "logon_count",
        "external_email_count",
        "usb_copy_count",
        "www_upload_count"
    ]
)

baseline_351 = pd.read_csv(
    SOURCE_PATHS_351["Original Baseline"],
    usecols=[
        "user",
        "email_count_p95",
        "usb_copy_p95",
        "http_upload_p95"
    ]
)

catalog_351 = pd.read_parquet(
    SOURCE_PATHS_351["Approved Scenario Catalog"],
    columns=[
        "scenario_id",
        "stealth_level",
        "stealth_multiplier"
    ]
)

schedule_351 = pd.read_parquet(
    SOURCE_PATHS_351["Validated Schedule"]
)

saved_users_351 = pd.read_parquet(
    OUTPUT_PATHS_351["GT Users"]
)

saved_channels_351 = pd.read_parquet(
    OUTPUT_PATHS_351["GT Channel-Days"]
)

saved_days_351 = pd.read_parquet(
    OUTPUT_PATHS_351["GT User-Days"]
)

saved_features_351 = pd.read_parquet(
    OUTPUT_PATHS_351["Final Features"]
)

KEYS_351 = ["user", "day"]


# ------------------------------------------------------------
# 4. Inspect original date precisions, then normalize
#    the independent in-memory copies to datetime64[ns].
#
# This resolves datetime64[s] vs datetime64[ms].
# It does NOT alter source files or calendar dates.
# ------------------------------------------------------------

date_frames_351 = {
    "Original User-Day": raw_351,
    "Validated Schedule": schedule_351,
    "Saved GT Channel-Days": saved_channels_351,
    "Saved GT User-Days": saved_days_351,
    "Saved Features": saved_features_351
}

date_precision_rows_351 = []

for name, frame in date_frames_351.items():

    original_dtype = str(frame["day"].dtype)

    parsed_dates = pd.to_datetime(
        frame["day"],
        errors="raise"
    )

    # Explicitly cast all datetimes to nanosecond
    # precision, then normalize to midnight.
    normalized_dates = (
        parsed_dates
        .astype("datetime64[ns]")
        .dt.normalize()
    )

    # Independently verify that calendar dates are
    # unchanged by the normalization.
    original_calendar = (
        parsed_dates.dt.strftime("%Y-%m-%d")
    )

    normalized_calendar = (
        normalized_dates.dt.strftime("%Y-%m-%d")
    )

    assert original_calendar.equals(
        normalized_calendar
    ), f"Calendar date changed: {name}"

    frame["day"] = normalized_dates

    assert str(frame["day"].dtype) == (
        "datetime64[ns]"
    )

    date_precision_rows_351.append({
        "Dataset": name,
        "Original_Dtype": original_dtype,
        "Normalized_Dtype": str(frame["day"].dtype),
        "Calendar_Dates_Unchanged": True
    })

print("\nB. DATE PRECISION NORMALIZATION")

date_precision_351 = pd.DataFrame(
    date_precision_rows_351
)

display(date_precision_351)

print("Date precision normalization: PASS")
print("Calendar dates preserved: PASS")


# ------------------------------------------------------------
# 5. Validate data structure and primary keys.
# ------------------------------------------------------------

assert len(raw_351) == 1394010
assert len(schedule_351) == 15010
assert len(saved_users_351) == 670
assert len(saved_channels_351) == 14375
assert len(saved_days_351) == 69634
assert saved_features_351.shape == (69634, 115)

assert baseline_351["user"].is_unique
assert catalog_351["scenario_id"].is_unique
assert saved_users_351["user"].is_unique

for name, frame in [
    ("Original User-Day", raw_351),
    ("Validated Schedule", schedule_351),
    ("Saved GT User-Days", saved_days_351),
    ("Saved Features", saved_features_351)
]:

    assert not frame.duplicated(
        KEYS_351
    ).any(), f"Duplicate user-day keys: {name}"

assert not saved_channels_351.duplicated(
    ["user", "day", "channel"]
).any()

print("\nC. FILE LOADING AND STRUCTURAL CHECKS: PASS")


# ------------------------------------------------------------
# 6. Independently reconstruct eligible user-days.
# ------------------------------------------------------------

simulated_users_351 = set(
    saved_users_351["user"]
)

assert len(simulated_users_351) == 670

eligible_351 = raw_351.loc[
    raw_351["user"].isin(simulated_users_351)
    & raw_351["day"].ge(pd.Timestamp("2011-01-01"))
    & raw_351["is_weekend"].eq(0)
    & raw_351["logon_count"].gt(0)
].copy()

eligible_351 = (
    eligible_351
    .sort_values(KEYS_351)
    .reset_index(drop=True)
)

assert len(eligible_351) == 69634
assert eligible_351["user"].nunique() == 670

reference_keys_351 = (
    eligible_351[KEYS_351]
    .reset_index(drop=True)
)

print("\nD. INDEPENDENT ELIGIBLE USER-DAY KEYS")

key_check_rows_351 = []

for name, frame in [
    ("GT User-Days", saved_days_351),
    ("Final Features", saved_features_351)
]:

    actual_keys = (
        frame[KEYS_351]
        .sort_values(KEYS_351)
        .reset_index(drop=True)
    )

    # This now compares the same datetime64[ns] dtype.
    pd.testing.assert_frame_equal(
        reference_keys_351,
        actual_keys,
        check_dtype=True,
        check_exact=True,
        obj=name
    )

    key_check_rows_351.append({
        "Dataset": name,
        "Expected_Keys": len(reference_keys_351),
        "Actual_Keys": len(actual_keys),
        "Missing_Keys": 0,
        "Unexpected_Keys": 0,
        "Status": "PASS"
    })

key_audit_351 = pd.DataFrame(
    key_check_rows_351
)

display(key_audit_351)

print("Independent eligible user-day keys: PASS")


# ------------------------------------------------------------
# 7. Independently expand the original validated schedule.
# ------------------------------------------------------------

planned_351 = schedule_351[
    [
        "user",
        "day",
        "scenario_id",
        "planned_channels",
        "attack_channels"
    ]
].copy()


def parse_channels_351(value):

    if pd.isna(value):
        return []

    return [
        item.strip().lower()
        for item in str(value).split("|")
        if item.strip()
    ]


planned_351["channels_list"] = (
    planned_351["attack_channels"]
    .map(parse_channels_351)
)

assert (
    planned_351["channels_list"].map(len)
    == planned_351["planned_channels"]
).all()

assert planned_351["channels_list"].map(
    lambda x: len(x) == len(set(x))
).all()

planned_351 = (
    planned_351
    .explode("channels_list")
    .rename(columns={
        "channels_list": "channel"
    })
)

planned_351 = planned_351.loc[
    planned_351["channel"].notna(),
    ["user", "day", "scenario_id", "channel"]
].copy()

assert len(planned_351) == 14375

assert set(planned_351["channel"]) == {
    "email", "usb", "http"
}

assert not planned_351.duplicated(
    ["user", "day", "channel"]
).any()

print("\nE. INDEPENDENT SCHEDULE EXPANSION: PASS")
print("Scheduled channel-days:", len(planned_351))


# ------------------------------------------------------------
# 8. Reconstruct natural counts and historical P95
#    from ORIGINAL source datasets.
# ------------------------------------------------------------

expected_351 = planned_351.merge(
    eligible_351[
        KEYS_351 + [
            "external_email_count",
            "usb_copy_count",
            "www_upload_count"
        ]
    ],
    on=KEYS_351,
    how="left",
    validate="many_to_one",
    indicator=True
)

assert expected_351["_merge"].eq("both").all()

expected_351.drop(
    columns=["_merge"],
    inplace=True
)

expected_351 = expected_351.merge(
    baseline_351,
    on="user",
    how="left",
    validate="many_to_one"
)

expected_351 = expected_351.merge(
    catalog_351,
    on="scenario_id",
    how="left",
    validate="many_to_one"
)

COUNT_MAP_351 = {
    "email": "external_email_count",
    "usb": "usb_copy_count",
    "http": "www_upload_count"
}

P95_MAP_351 = {
    "email": "email_count_p95",
    "usb": "usb_copy_p95",
    "http": "http_upload_p95"
}

expected_351["natural_count"] = np.nan
expected_351["historical_p95"] = np.nan

for channel in COUNT_MAP_351:

    mask = expected_351["channel"].eq(channel)

    expected_351.loc[
        mask, "natural_count"
    ] = expected_351.loc[
        mask, COUNT_MAP_351[channel]
    ].to_numpy(dtype=float)

    expected_351.loc[
        mask, "historical_p95"
    ] = expected_351.loc[
        mask, P95_MAP_351[channel]
    ].to_numpy(dtype=float)

assert expected_351["natural_count"].notna().all()
assert expected_351["historical_p95"].notna().all()

assert expected_351["natural_count"].gt(0).all()
assert expected_351["historical_p95"].gt(0).all()

assert expected_351[
    ["stealth_level", "stealth_multiplier"]
].notna().all().all()

MULTIPLIERS_351 = {
    "High": 0.5,
    "Medium": 1.0,
    "Low": 1.5
}

np.testing.assert_allclose(
    expected_351["stealth_multiplier"],
    expected_351["stealth_level"].map(
        MULTIPLIERS_351
    ),
    rtol=0,
    atol=1e-12
)

print("\nF. SOURCE VALUES AND BASELINE VALIDATION: PASS")


# ------------------------------------------------------------
# 9. Independently recalculate Formula B.
#
# Formula B:
# max(1, ceil(Natural Count * Stealth Multiplier))
# ------------------------------------------------------------

expected_351["injected_count"] = np.maximum(
    1,
    np.ceil(
        expected_351["natural_count"].to_numpy(
            dtype=float
        )
        * expected_351["stealth_multiplier"].to_numpy(
            dtype=float
        )
    )
).astype(np.int64)

expected_351["final_count"] = (
    expected_351["natural_count"]
    + expected_351["injected_count"]
)

independent_total_351 = int(
    expected_351["injected_count"].sum()
)

print("\nG. INDEPENDENT FORMULA B RESULTS")

independent_summary_351 = (
    expected_351
    .groupby(["channel", "stealth_level"])
    .agg(
        Channel_Days=("user", "size"),
        Added_Events=("injected_count", "sum")
    )
    .reset_index()
)

display(independent_summary_351)

print(
    "Independently recalculated injected events:",
    independent_total_351
)


# ------------------------------------------------------------
# 10. Compare all recalculated Channel-Day values against
#     the SAVED Ground Truth file.
#
# Report mismatch counts, not just PASS/FAIL.
# ------------------------------------------------------------

print("\nH. CHANNEL-DAY VALUE-BY-VALUE COMPARISON")

JOIN_KEYS_351 = ["user", "day", "channel"]

saved_channels_351["channel"] = (
    saved_channels_351["channel"]
    .astype(str)
    .str.strip()
    .str.lower()
)

expected_compare_351 = expected_351[
    JOIN_KEYS_351 + [
        "scenario_id",
        "stealth_level",
        "natural_count",
        "historical_p95",
        "stealth_multiplier",
        "injected_count",
        "final_count"
    ]
].copy()

comparison_351 = expected_compare_351.merge(
    saved_channels_351,
    on=JOIN_KEYS_351,
    how="outer",
    suffixes=("_expected", "_saved"),
    indicator=True,
    validate="one_to_one"
)

matched_351 = comparison_351.loc[
    comparison_351["_merge"].eq("both")
].copy()

audit_rows_351 = [
    {
        "Field": "Missing saved channel-days",
        "Mismatches": int(
            comparison_351["_merge"].eq("left_only").sum()
        )
    },
    {
        "Field": "Unexpected saved channel-days",
        "Mismatches": int(
            comparison_351["_merge"].eq("right_only").sum()
        )
    }
]

mismatch_examples_351 = {}

# Independently compare all numeric fields.
for field in [
    "natural_count",
    "historical_p95",
    "stealth_multiplier",
    "injected_count",
    "final_count"
]:

    expected_values = (
        matched_351[f"{field}_expected"]
        .to_numpy(dtype=float)
    )

    saved_values = (
        matched_351[f"{field}_saved"]
        .to_numpy(dtype=float)
    )

    equal = np.isclose(
        expected_values,
        saved_values,
        rtol=0,
        atol=1e-10,
        equal_nan=False
    )

    mismatches = int((~equal).sum())

    audit_rows_351.append({
        "Field": field,
        "Mismatches": mismatches
    })

    if mismatches > 0:

        mismatch_examples_351[field] = (
            matched_351.loc[
                ~equal,
                JOIN_KEYS_351 + [
                    f"{field}_expected",
                    f"{field}_saved"
                ]
            ].head(10)
        )

# Compare scenario IDs and stealth labels.
for field in ["scenario_id", "stealth_level"]:

    equal = (
        matched_351[f"{field}_expected"]
        .eq(
            matched_351[f"{field}_saved"]
        )
        .fillna(False)
    )

    mismatches = int((~equal).sum())

    audit_rows_351.append({
        "Field": field,
        "Mismatches": mismatches
    })

    if mismatches > 0:

        mismatch_examples_351[field] = (
            matched_351.loc[
                ~equal,
                JOIN_KEYS_351 + [
                    f"{field}_expected",
                    f"{field}_saved"
                ]
            ].head(10)
        )

formula_mismatches_351 = int(
    (~saved_channels_351[
        "injection_formula"
    ].eq("B_Natural_Addition")).sum()
)

audit_rows_351.append({
    "Field": "injection_formula",
    "Mismatches": formula_mismatches_351
})

channel_audit_351 = pd.DataFrame(
    audit_rows_351
)

display(channel_audit_351)

if mismatch_examples_351:

    print("\nCHANNEL-DAY MISMATCH EXAMPLES")

    for field, examples in mismatch_examples_351.items():

        print(f"\nField: {field}")
        display(examples)


# ------------------------------------------------------------
# 11. Independently reconstruct ALL final daily counts
#     from original User-Day + recalculated injections.
#
# This includes every non-injected day as well.
# ------------------------------------------------------------

print("\nI. INDEPENDENT FINAL DAILY COUNT COMPARISON")

daily_injections_351 = (
    expected_351
    .pivot_table(
        index=KEYS_351,
        columns="channel",
        values="injected_count",
        aggfunc="sum",
        fill_value=0
    )
    .rename(columns={
        "email": "add_email",
        "usb": "add_usb",
        "http": "add_http"
    })
    .reset_index()
)

reference_daily_351 = eligible_351[
    KEYS_351 + list(COUNT_MAP_351.values())
].merge(
    daily_injections_351,
    on=KEYS_351,
    how="left",
    validate="one_to_one"
)

ADD_COLS_351 = [
    "add_email",
    "add_usb",
    "add_http"
]

reference_daily_351[ADD_COLS_351] = (
    reference_daily_351[ADD_COLS_351]
    .fillna(0)
)

daily_compare_351 = reference_daily_351.merge(
    saved_features_351[
        KEYS_351 + list(COUNT_MAP_351.values())
    ],
    on=KEYS_351,
    how="outer",
    suffixes=("_original", "_saved"),
    indicator=True,
    validate="one_to_one"
)

daily_missing_351 = int(
    daily_compare_351["_merge"].ne("both").sum()
)

daily_audit_rows_351 = []

for channel, source_col in COUNT_MAP_351.items():

    add_col = f"add_{channel}"

    expected_count = (
        daily_compare_351[
            f"{source_col}_original"
        ].to_numpy(dtype=float)
        + daily_compare_351[
            add_col
        ].to_numpy(dtype=float)
    )

    saved_count = (
        daily_compare_351[
            f"{source_col}_saved"
        ].to_numpy(dtype=float)
    )

    equal = np.isclose(
        expected_count,
        saved_count,
        rtol=0,
        atol=1e-10,
        equal_nan=False
    )

    mismatches = int((~equal).sum())

    daily_audit_rows_351.append({
        "Channel": channel.upper(),
        "Compared_Rows": len(equal),
        "Mismatched_Rows": mismatches,
        "Expected_Injected": int(
            daily_compare_351[add_col].sum()
        )
    })

    if mismatches > 0:

        print(f"\nMismatch examples: {channel}")

        display(
            daily_compare_351.loc[
                ~equal,
                KEYS_351 + [
                    f"{source_col}_original",
                    add_col,
                    f"{source_col}_saved"
                ]
            ].head(10)
        )

daily_audit_351 = pd.DataFrame(
    daily_audit_rows_351
)

display(daily_audit_351)


# ------------------------------------------------------------
# 12. Final independent audit decision.
# ------------------------------------------------------------

total_channel_mismatches_351 = int(
    channel_audit_351["Mismatches"].sum()
)

total_daily_mismatches_351 = (
    daily_missing_351
    + int(
        daily_audit_351["Mismatched_Rows"].sum()
    )
)

print("\n" + "=" * 80)
print("STAGE 3.5 — STEP 3.5.1: AUDIT RESULTS")
print("=" * 80)

print("Original sources loaded from disk: YES")
print("Validated schedule loaded from disk: YES")
print("Saved outputs loaded from disk: YES")
print("All date columns normalized to datetime64[ns]: YES")

print(
    "Recalculated channel-days:",
    len(expected_351)
)

print(
    "Reconstructed eligible user-days:",
    len(eligible_351)
)

print(
    "Independent injected events:",
    independent_total_351
)

print(
    "Channel GT mismatches:",
    total_channel_mismatches_351
)

print(
    "Daily count mismatches:",
    total_daily_mismatches_351
)

print("Original files modified: NO")
print("Output files modified: NO")

assert independent_total_351 == 42095, (
    "Independent total differs from the expected injection."
)

assert total_channel_mismatches_351 == 0, (
    "Channel Ground Truth mismatch detected."
)

assert total_daily_mismatches_351 == 0, (
    "Final daily activity mismatch detected."
)

print("\nSTEP 3.5.1 RESULT: PASS")
print("Independent Formula B calculation: VERIFIED")
print("Saved channel-level Ground Truth: VERIFIED")
print("Saved Email/USB/HTTP daily counts: VERIFIED")
print("Remaining Stage 3.5 checks: NOT YET PERFORMED")


STAGE 3.5 — STEP 3.5.1
INDEPENDENT INJECTION RECALCULATION

A. FILE DISCOVERY: PASS


,Dataset,Size_MB,SHA256
0,Original User-Day,23.507,bd988e53112cd5a4b328eb6fde5ada78d2962d5b930357...
1,Original Baseline,0.885,d750760f9a62c6e133cfd81b19930f6193066a31f0fd3f...
2,Approved Scenario Catalog,0.024,13b815102abe1b1d53195f23fc0ffe6a78e6ddb99dc5b4...
3,Validated Schedule,0.064,5bb4c5619d0d9e90a13bd24333ae883951aece2ad87c08...
4,GT Users,0.021,f0d4aeaa74762724c7775d3b21b87707997c767b8848d2...
5,GT Channel-Days,0.069,52ae9eb047a7d9b5d30908ba33131f0a2ceb5675854fa6...
6,GT User-Days,0.088,4741e719274b81da1ff8cce29ce6c52d4360d3742d7213...
7,Final Features,7.035,90e9fb58be4bb2ff61a17397425816a11453a00c493494...



B. DATE PRECISION NORMALIZATION


,Dataset,Original_Dtype,Normalized_Dtype,Calendar_Dates_Unchanged
0,Original User-Day,object,datetime64[ns],True
1,Validated Schedule,datetime64[ns],datetime64[ns],True
2,Saved GT Channel-Days,datetime64[ns],datetime64[ns],True
3,Saved GT User-Days,datetime64[ms],datetime64[ns],True
4,Saved Features,datetime64[ms],datetime64[ns],True


Date precision normalization: PASS
Calendar dates preserved: PASS

C. FILE LOADING AND STRUCTURAL CHECKS: PASS

D. INDEPENDENT ELIGIBLE USER-DAY KEYS


,Dataset,Expected_Keys,Actual_Keys,Missing_Keys,Unexpected_Keys,Status
0,GT User-Days,69634,69634,0,0,PASS
1,Final Features,69634,69634,0,0,PASS


... (3 outputs left out)

,Field,Mismatches
0,Missing saved channel-days,0
1,Unexpected saved channel-days,0
2,natural_count,0
3,historical_p95,0
4,stealth_multiplier,0
5,injected_count,0
6,final_count,0
7,scenario_id,0
8,stealth_level,0
9,injection_formula,0



I. INDEPENDENT FINAL DAILY COUNT COMPARISON


,Channel,Compared_Rows,Mismatched_Rows,Expected_Injected
0,EMAIL,69634,0,23880
1,USB,69634,0,11422
2,HTTP,69634,0,6793



STAGE 3.5 — STEP 3.5.1: AUDIT RESULTS
Original sources loaded from disk: YES
Validated schedule loaded from disk: YES
Saved outputs loaded from disk: YES
All date columns normalized to datetime64[ns]: YES
Recalculated channel-days: 14375
Reconstructed eligible user-days: 69634
Independent injected events: 42095
Channel GT mismatches: 0
Daily count mismatches: 0
Original files modified: NO
Output files modified: NO

STEP 3.5.1 RESULT: PASS
Independent Formula B calculation: VERIFIED
Saved channel-level Ground Truth: VERIFIED
Saved Email/USB/HTTP daily counts: VERIFIED
Remaining Stage 3.5 checks: NOT YET PERFORMED



### Step 3.5.2 — Independent Scheduling & Scenario Validation

**Objective:** Independently verify that every simulated attack schedule satisfies the original approved scenario requirements and natural activity constraints.

**Independent Sources:**
- Original CERT r6.2 User-Day dataset
- Approved Scenario Catalog
- Validated schedule reference exported from Stage 3.3.9

**Outputs Under Test:**
- Saved User-Level Ground Truth
- Saved User-Day Ground Truth

**Validation Tasks:**

1. Validate the mathematical consistency of each scenario:
   - `Duration = Active Days + Off Days`
   - `Active Days = Single + Two + Three Channel Days`
   - `0 <= Target Switches <= Active Days - 1`

2. Reconstruct each user's eligible workdays from the original User-Day data.

3. Verify that the scheduled days exactly match the eligible workdays within the selected attack window.

4. Independently count Off, Single, Two, and Three-Channel days.

5. Verify that every selected attack channel had positive natural activity on that exact day.

6. Verify that the primary channel belongs to the selected attack channels.

7. Recalculate primary-channel switches chronologically, ignoring Off days when counting transitions.

8. Compare duration, attack days, off days, switching counts, scenario IDs, stealth levels, and attack-window dates against saved User-Level Ground Truth.

9. Report all mathematical and logical discrepancies by user and check type.

**Important Distinction:**

Primary-channel switches are part of the synthetic scheduling Ground Truth. They are not the same as the observed dominant-channel switches calculated in the behavioral feature dataset.

**Independence:** Do not use previously generated scheduling validation results, saved switch totals, or injected counts to derive expected scenario values.

**Safety:** Read-only. No source files, schedules, or exported outputs are modified.

**Expected Outcome:** Independent verification of all 670 schedules, or a detailed discrepancy report.


In [4]:

# Stage 3 — Step 3.5.2: Independent Scheduling & Scenario Validation
# Goal: Independently verify 670 attack schedules against
# original scenario requirements and original daily activity.
#
# READ-ONLY:
# No original data, Ground Truth, or files modified.
# No reliance on previous scheduling validation results.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 80)
print("STAGE 3.5 — STEP 3.5.2")
print("INDEPENDENT SCHEDULING & SCENARIO VALIDATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Reload the complete approved scenario catalog.
#    Step 3.5.1 loaded only the stealth-related columns.
# ------------------------------------------------------------

assert "SOURCE_PATHS_351" in globals()
assert "schedule_351" in globals()
assert "eligible_351" in globals()
assert "saved_users_351" in globals()
assert "saved_days_351" in globals()

catalog_352 = pd.read_parquet(
    SOURCE_PATHS_351["Approved Scenario Catalog"]
)

schedule_352 = schedule_351.copy(deep=True)
eligible_352 = eligible_351.copy(deep=True)
users_gt_352 = saved_users_351.copy(deep=True)
days_gt_352 = saved_days_351.copy(deep=True)

KEYS_352 = ["user", "day"]

REQUIRED_CATALOG_352 = [
    "scenario_id",
    "duration",
    "active_days",
    "off_days",
    "single_channel_days",
    "two_channel_days",
    "three_channel_days",
    "target_switches",
    "stealth_level"
]

REQUIRED_SCHEDULE_352 = [
    "user",
    "day",
    "scenario_id",
    "window_start",
    "window_end",
    "planned_channels",
    "attack_channels",
    "primary_channel"
]

REQUIRED_USER_GT_352 = [
    "user",
    "scenario_id",
    "stealth_level",
    "attack_start",
    "attack_end",
    "duration_days",
    "attack_days",
    "off_days",
    "planned_switches"
]

for name, frame, columns in [
    ("Catalog", catalog_352, REQUIRED_CATALOG_352),
    ("Schedule", schedule_352, REQUIRED_SCHEDULE_352),
    ("User GT", users_gt_352, REQUIRED_USER_GT_352)
]:
    missing = sorted(set(columns) - set(frame.columns))
    assert not missing, f"{name} missing columns: {missing}"

assert catalog_352["scenario_id"].is_unique
assert users_gt_352["user"].is_unique
assert not schedule_352.duplicated(KEYS_352).any()

# Normalize dates in working copies only.
for column in ["day", "window_start", "window_end"]:
    schedule_352[column] = (
        pd.to_datetime(
            schedule_352[column],
            errors="raise"
        )
        .astype("datetime64[ns]")
        .dt.normalize()
    )

for column in ["attack_start", "attack_end"]:
    users_gt_352[column] = (
        pd.to_datetime(
            users_gt_352[column],
            errors="raise"
        )
        .astype("datetime64[ns]")
        .dt.normalize()
    )

eligible_352["day"] = (
    pd.to_datetime(eligible_352["day"])
    .astype("datetime64[ns]")
    .dt.normalize()
)

days_gt_352["day"] = (
    pd.to_datetime(days_gt_352["day"])
    .astype("datetime64[ns]")
    .dt.normalize()
)

assert len(schedule_352) == 15010
assert schedule_352["user"].nunique() == 670
assert len(users_gt_352) == 670

print("\nA. INPUT STRUCTURE: PASS")
print("Approved catalog scenarios:", len(catalog_352))
print("Scheduled users:", schedule_352["user"].nunique())
print("Scheduled days:", len(schedule_352))

# ------------------------------------------------------------
# 2. Independently verify catalog mathematical identities.
# ------------------------------------------------------------

catalog_checks_352 = catalog_352.copy()

catalog_checks_352["valid_duration"] = (
    catalog_checks_352["duration"]
    ==
    catalog_checks_352["active_days"]
    + catalog_checks_352["off_days"]
)

catalog_checks_352["valid_channel_days"] = (
    catalog_checks_352["active_days"]
    ==
    catalog_checks_352["single_channel_days"]
    + catalog_checks_352["two_channel_days"]
    + catalog_checks_352["three_channel_days"]
)

catalog_checks_352["valid_switch_target"] = (
    catalog_checks_352["target_switches"].ge(0)
    &
    catalog_checks_352["target_switches"].le(
        catalog_checks_352["active_days"] - 1
    )
)

catalog_checks_352["valid_nonnegative_days"] = (
    catalog_checks_352[
        [
            "active_days",
            "off_days",
            "single_channel_days",
            "two_channel_days",
            "three_channel_days"
        ]
    ].ge(0).all(axis=1)
)

catalog_summary_352 = pd.DataFrame([
    {
        "Check": column,
        "Scenarios_Checked": len(catalog_checks_352),
        "Failures": int(
            (~catalog_checks_352[column]).sum()
        )
    }
    for column in [
        "valid_duration",
        "valid_channel_days",
        "valid_switch_target",
        "valid_nonnegative_days"
    ]
])

print("\nB. SCENARIO CATALOG MATHEMATICAL VALIDATION")
display(catalog_summary_352)

# ------------------------------------------------------------
# 3. Build independent lookups.
# ------------------------------------------------------------

rules_352 = catalog_352.set_index("scenario_id")

gt_users_lookup_352 = users_gt_352.set_index("user")

eligible_groups_352 = {
    user: group.sort_values("day").reset_index(drop=True)
    for user, group in eligible_352.groupby("user")
}

CHANNEL_COLUMNS_352 = {
    "Email": "external_email_count",
    "USB": "usb_copy_count",
    "HTTP": "www_upload_count"
}

VALID_CHANNELS_352 = set(CHANNEL_COLUMNS_352)

def parse_channels_352(value):

    if pd.isna(value) or str(value).strip() == "":
        return []

    return [
        part.strip()
        for part in str(value).split("|")
        if part.strip()
    ]


def compare_integer_352(a, b):
    if pd.isna(a) or pd.isna(b):
        return False
    return int(a) == int(b)


# ------------------------------------------------------------
# 4. Independently validate each user schedule.
#
# Do not use validated_users_339 or previous calculated
# switching results.
# ------------------------------------------------------------

user_results_352 = []
issue_rows_352 = []

for user, group in schedule_352.groupby("user"):

    group = (
        group.sort_values("day")
        .reset_index(drop=True)
    )

    issues = []

    scenario_ids = group["scenario_id"].dropna().unique()

    if len(scenario_ids) != 1:
        issues.append("Multiple or missing scenario IDs")
        continue

    scenario_id = scenario_ids[0]

    if scenario_id not in rules_352.index:
        issues.append("Scenario missing from approved catalog")
        continue

    if user not in gt_users_lookup_352.index:
        issues.append("User missing from saved User GT")
        continue

    if user not in eligible_groups_352:
        issues.append("No original eligible workdays")
        continue

    rule = rules_352.loc[scenario_id]
    gt = gt_users_lookup_352.loc[user]

    # --------------------------------------------------------
    # 4.1 Validate date-window metadata.
    # --------------------------------------------------------

    unique_starts = group["window_start"].unique()
    unique_ends = group["window_end"].unique()

    valid_window_metadata = (
        len(unique_starts) == 1
        and len(unique_ends) == 1
    )

    if not valid_window_metadata:
        issues.append("Inconsistent window boundaries")
        start_date = pd.NaT
        end_date = pd.NaT
    else:
        start_date = pd.Timestamp(unique_starts[0])
        end_date = pd.Timestamp(unique_ends[0])

        if start_date > end_date:
            issues.append("Window start after window end")

    # --------------------------------------------------------
    # 4.2 Independently reconstruct eligible days
    #     within the scheduled attack window.
    # --------------------------------------------------------

    expected_days = []

    if valid_window_metadata and start_date <= end_date:

        expected_days = (
            eligible_groups_352[user]
            .loc[
                lambda d: d["day"].between(
                    start_date, end_date
                ),
                "day"
            ]
            .sort_values()
            .dt.date
            .tolist()
        )

    actual_days = (
        group["day"]
        .dt.date
        .tolist()
    )

    valid_day_sequence = (
        actual_days == expected_days
    )

    if not valid_day_sequence:
        issues.append(
            "Schedule differs from original eligible workdays"
        )

    # --------------------------------------------------------
    # 4.3 Recalculate day categories.
    # --------------------------------------------------------

    day_counts = (
        group["planned_channels"]
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    )

    observed_off = int(day_counts[0])
    observed_single = int(day_counts[1])
    observed_two = int(day_counts[2])
    observed_three = int(day_counts[3])

    observed_active = (
        observed_single
        + observed_two
        + observed_three
    )

    day_type_checks = {
        "Duration": compare_integer_352(
            len(group), rule["duration"]
        ),
        "Active": compare_integer_352(
            observed_active, rule["active_days"]
        ),
        "Off": compare_integer_352(
            observed_off, rule["off_days"]
        ),
        "Single": compare_integer_352(
            observed_single, rule["single_channel_days"]
        ),
        "Two": compare_integer_352(
            observed_two, rule["two_channel_days"]
        ),
        "Three": compare_integer_352(
            observed_three, rule["three_channel_days"]
        )
    }

    for name, passed in day_type_checks.items():
        if not passed:
            issues.append(
                f"{name} count does not match catalog"
            )

    # --------------------------------------------------------
    # 4.4 Independently verify selected channels against
    #     original daily activity and primary-channel rules.
    # --------------------------------------------------------

    original_daily = (
        eligible_groups_352[user]
        .set_index("day")
    )

    invalid_channel_days = 0
    invalid_primary_days = 0
    invalid_off_days = 0

    realized_switches = 0
    previous_primary = None

    for record in group.itertuples(index=False):

        selected = parse_channels_352(
            record.attack_channels
        )

        planned = int(record.planned_channels)

        if (
            len(selected) != planned
            or len(selected) != len(set(selected))
            or not set(selected).issubset(
                VALID_CHANNELS_352
            )
        ):
            invalid_channel_days += 1

        # Source data from the original eligible User-Day.
        if record.day not in original_daily.index:
            invalid_channel_days += 1
            continue

        natural_row = original_daily.loc[record.day]

        available = {
            channel
            for channel, count_col in CHANNEL_COLUMNS_352.items()
            if natural_row[count_col] > 0
        }

        if not set(selected).issubset(available):
            invalid_channel_days += 1

        if planned == 0:

            if (
                selected
                or pd.notna(record.primary_channel)
            ):
                invalid_off_days += 1

            # Off days do not reset the previous primary.
            continue

        primary = record.primary_channel

        if (
            not isinstance(primary, str)
            or primary not in selected
        ):
            invalid_primary_days += 1
            continue

        if (
            previous_primary is not None
            and primary != previous_primary
        ):
            realized_switches += 1

        previous_primary = primary

    if invalid_channel_days:
        issues.append("Invalid selected/available channels")

    if invalid_primary_days:
        issues.append("Invalid primary-channel assignments")

    if invalid_off_days:
        issues.append("Off-day channel/primary violation")

    expected_switches = int(
        rule["target_switches"]
    )

    switch_pass = (
        realized_switches == expected_switches
    )

    if not switch_pass:
        issues.append("Primary switch count mismatch")

    # --------------------------------------------------------
    # 4.5 Compare against saved User-Level Ground Truth.
    # --------------------------------------------------------

    metadata_checks = {
        "Scenario": (
            str(gt["scenario_id"]) == str(scenario_id)
        ),
        "Stealth": (
            str(gt["stealth_level"])
            == str(rule["stealth_level"])
        ),
        "Duration": compare_integer_352(
            gt["duration_days"], len(group)
        ),
        "Attack Days": compare_integer_352(
            gt["attack_days"], observed_active
        ),
        "Off Days": compare_integer_352(
            gt["off_days"], observed_off
        ),
        "Primary Switches": compare_integer_352(
            gt["planned_switches"],
            realized_switches
        ),
        "Attack Start": (
            valid_window_metadata
            and gt["attack_start"] == start_date
        ),
        "Attack End": (
            valid_window_metadata
            and gt["attack_end"] == end_date
        )
    }

    for name, passed in metadata_checks.items():
        if not passed:
            issues.append(
                f"User GT mismatch: {name}"
            )

    # --------------------------------------------------------
    # 4.6 Store independently calculated audit results.
    # --------------------------------------------------------

    result = {
        "user": user,
        "scenario_id": scenario_id,
        "stealth_level": rule["stealth_level"],
        "duration": len(group),
        "off_days": observed_off,
        "single_days": observed_single,
        "two_days": observed_two,
        "three_days": observed_three,
        "active_days": observed_active,
        "expected_switches": expected_switches,
        "realized_switches": realized_switches,
        "eligible_day_sequence_pass": valid_day_sequence,
        "invalid_channel_days": invalid_channel_days,
        "invalid_primary_days": invalid_primary_days,
        "invalid_off_days": invalid_off_days,
        "scenario_counts_pass": all(
            day_type_checks.values()
        ),
        "user_gt_metadata_pass": all(
            metadata_checks.values()
        ),
        "switches_pass": switch_pass,
        "issue_count": len(issues),
        "status": (
            "PASS" if not issues else "FAIL"
        )
    }

    user_results_352.append(result)

    for issue in issues:
        issue_rows_352.append({
            "user": user,
            "scenario_id": scenario_id,
            "issue": issue
        })

# ------------------------------------------------------------
# 5. Summarize every independent validation dimension.
# ------------------------------------------------------------

user_audit_352 = pd.DataFrame(user_results_352)
issues_352 = pd.DataFrame(
    issue_rows_352,
    columns=["user", "scenario_id", "issue"]
)

print("\nC. INDEPENDENT SCHEDULE VALIDATION SUMMARY")

check_columns_352 = [
    "eligible_day_sequence_pass",
    "scenario_counts_pass",
    "switches_pass",
    "user_gt_metadata_pass"
]

schedule_summary_352 = pd.DataFrame([
    {
        "Check": column,
        "Users_Checked": len(user_audit_352),
        "Passed": int(user_audit_352[column].sum()),
        "Failed": int((~user_audit_352[column]).sum())
    }
    for column in check_columns_352
])

display(schedule_summary_352)

print("\nD. PRIMARY CHANNEL SWITCH VALIDATION")

switch_summary_352 = (
    user_audit_352
    .groupby("stealth_level")
    .agg(
        Users=("user", "size"),
        Expected_Switches=("expected_switches", "sum"),
        Recalculated_Switches=("realized_switches", "sum"),
        Mismatched_Users=(
            "switches_pass",
            lambda s: int((~s).sum())
        )
    )
    .reindex(["High", "Medium", "Low"])
    .reset_index()
)

display(switch_summary_352)

# ------------------------------------------------------------
# 6. Compare independently calculated day totals.
# ------------------------------------------------------------

print("\nE. INDEPENDENT DAY-TYPE TOTALS")

day_totals_352 = pd.DataFrame([
    {
        "Day_Type": "Off",
        "Recalculated_Days": int(
            user_audit_352["off_days"].sum()
        ),
        "Expected_Days": 2374
    },
    {
        "Day_Type": "Single",
        "Recalculated_Days": int(
            user_audit_352["single_days"].sum()
        ),
        "Expected_Days": 11098
    },
    {
        "Day_Type": "Two",
        "Recalculated_Days": int(
            user_audit_352["two_days"].sum()
        ),
        "Expected_Days": 1337
    },
    {
        "Day_Type": "Three",
        "Recalculated_Days": int(
            user_audit_352["three_days"].sum()
        ),
        "Expected_Days": 201
    }
])

day_totals_352["Difference"] = (
    day_totals_352["Recalculated_Days"]
    - day_totals_352["Expected_Days"]
)

display(day_totals_352)

# ------------------------------------------------------------
# 7. Report any mismatches with user identifiers.
# ------------------------------------------------------------

print("\nF. MATHEMATICAL AND LOGICAL DISCREPANCIES")

catalog_failures_352 = int(
    catalog_summary_352["Failures"].sum()
)

schedule_failures_352 = int(
    user_audit_352["status"].ne("PASS").sum()
)

channel_violations_352 = int(
    user_audit_352["invalid_channel_days"].sum()
)

primary_violations_352 = int(
    user_audit_352["invalid_primary_days"].sum()
)

off_violations_352 = int(
    user_audit_352["invalid_off_days"].sum()
)

if issues_352.empty:
    print("No user-level discrepancies found.")
else:
    display(
        issues_352
        .groupby("issue")
        .size()
        .reset_index(name="Occurrences")
        .sort_values("Occurrences", ascending=False)
    )

    print("\nFirst 30 discrepancy examples:")
    display(issues_352.head(30))

print("\n" + "=" * 80)
print("STEP 3.5.2 — INDEPENDENT SCHEDULING AUDIT RESULTS")
print("=" * 80)

print("Catalog mathematical failures:", catalog_failures_352)
print("User schedules checked:", len(user_audit_352))
print("User schedule failures:", schedule_failures_352)
print("Invalid selected channel-days:", channel_violations_352)
print("Invalid primary-channel days:", primary_violations_352)
print("Invalid Off days:", off_violations_352)
print("Total independent scenario switches:",
      int(user_audit_352["realized_switches"].sum()))

print("Original files modified: NO")
print("Ground Truth files modified: NO")

# Validate the complete expected population.
assert len(user_audit_352) == 670, (
    "Not all 670 schedules were independently audited."
)

assert catalog_failures_352 == 0, (
    "Approved scenario catalog mathematical failures."
)

assert schedule_failures_352 == 0, (
    "Independent schedule validation FAILED."
)

assert day_totals_352["Difference"].eq(0).all(), (
    "Recalculated global day-type totals differ."
)

print("\nSTEP 3.5.2 RESULT: PASS")
print("Scenario catalog mathematics: VERIFIED")
print("Eligible-day sequences: VERIFIED")
print("Daily channel availability: VERIFIED")
print("Primary-channel switches: VERIFIED")
print("Saved User-Level Ground Truth: VERIFIED")
print("Remaining Stage 3.5 checks: NOT YET PERFORMED")


STAGE 3.5 — STEP 3.5.2
INDEPENDENT SCHEDULING & SCENARIO VALIDATION

A. INPUT STRUCTURE: PASS
Approved catalog scenarios: 1143
Scheduled users: 670
Scheduled days: 15010

B. SCENARIO CATALOG MATHEMATICAL VALIDATION


,Check,Scenarios_Checked,Failures
0,valid_duration,1143,0
1,valid_channel_days,1143,0
2,valid_switch_target,1143,0
3,valid_nonnegative_days,1143,0



C. INDEPENDENT SCHEDULE VALIDATION SUMMARY


,Check,Users_Checked,Passed,Failed
0,eligible_day_sequence_pass,670,670,0
1,scenario_counts_pass,670,670,0
2,switches_pass,670,670,0
3,user_gt_metadata_pass,670,670,0



D. PRIMARY CHANNEL SWITCH VALIDATION


,stealth_level,Users,Expected_Switches,Recalculated_Switches,Mismatched_Users
0,High,217,1309,1309,0
1,Medium,226,1551,1551,0
2,Low,227,1519,1519,0



E. INDEPENDENT DAY-TYPE TOTALS


,Day_Type,Recalculated_Days,Expected_Days,Difference
0,Off,2374,2374,0
1,Single,11098,11098,0
2,Two,1337,1337,0
3,Three,201,201,0



F. MATHEMATICAL AND LOGICAL DISCREPANCIES
No user-level discrepancies found.

STEP 3.5.2 — INDEPENDENT SCHEDULING AUDIT RESULTS
Catalog mathematical failures: 0
User schedules checked: 670
User schedule failures: 0
Invalid selected channel-days: 0
Invalid primary-channel days: 0
Invalid Off days: 0
Total independent scenario switches: 4379
Original files modified: NO
Ground Truth files modified: NO

STEP 3.5.2 RESULT: PASS
Scenario catalog mathematics: VERIFIED
Eligible-day sequences: VERIFIED
Daily channel availability: VERIFIED
Primary-channel switches: VERIFIED
Saved User-Level Ground Truth: VERIFIED
Remaining Stage 3.5 checks: NOT YET PERFORMED



### Step 3.5.3 — Independent Recalculation of the Original 85 Columns

**Objective:** Independently reconstruct the original 85-column behavioral feature schema after applying Formula B, and compare every reconstructed value against the saved final feature dataset.

**Independent Inputs:**
- Original CERT r6.2 User-Day dataset
- Original User Baseline
- Approved Scenario Catalog
- Validated Attack Schedule

**Output Under Test:**
`behavior_features_injected_final.parquet`

**Reconstruction:**
1. Independently select the 670 users' eligible workdays.
2. Recalculate Formula B additions from original daily activity.
3. Apply additions to all related aggregate event-count fields.
4. Preserve unrelated daily context, bytes, and historical baselines.
5. Recalculate daily channel activity indicators.
6. Recalculate P95 exceedance flags and ratios.
7. Recalculate the original 5/10/20 WD active-day features.
8. Recalculate original 5/10 WD channel-count totals.
9. Recalculate multi-channel activity days over 10 WD.

**Rolling-Window Semantics:**

The original Data Engineering features use partial causal windows (`min_periods=1`), including the first eligible workdays.

**Validation:**
- Match the original 85-column schema.
- Compare every non-key field on every one of 69,634 user-days.
- Verify null placement and timestamp values.
- Verify the four aggregate event-count identities.
- Report mismatches per column, including examples and maximum numerical error.
- Report independent verification coverage by feature category.

**Independence Rule:** Do not use previously generated injected User-Day variants or feature-rebuilding functions.

**Scope Limitation:** Daily source measurements and historical baseline values are independently reloaded and compared, but raw Email/File/HTTP events are not reconstructed in this step.

**Safety:** Read-only; no original or exported files are modified.

**Expected Outcome:** Zero unexplained value mismatches across the original 85-column schema.


In [5]:

# Stage 3 — Step 3.5.3: Independent Original Feature Rebuild
# Goal: Independently reconstruct all original 85 columns
# and compare every cell with the final saved dataset.
#
# STRICT READ-ONLY:
# No changes to source, Ground Truth, or feature files.

from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
from IPython.display import display

print("=" * 80)
print("STAGE 3.5 — STEP 3.5.3")
print("INDEPENDENT ORIGINAL 85-COLUMN FEATURE VALIDATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Resolve source files independently.
# ------------------------------------------------------------

ROOT_353 = Path(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration"
)

DE_353 = ROOT_353 / "Data_Engineering" / "Data_engineering"
ATTACK_353 = ROOT_353 / "Attack_Simulation"

PATHS_353 = {
    "user_day": (
        DE_353 / "data" / "processed"
        / "user_day_r62_v1.parquet"
    ),
    "baseline_parquet": (
        DE_353 / "data" / "features"
        / "user_baseline_r62_v1.parquet"
    ),
    "baseline_csv": (
        DE_353 / "data" / "features"
        / "user_baseline_r62_v1.csv"
    ),
    "catalog": (
        ATTACK_353 / "data"
        / "approved_scenario_catalog_r62.parquet"
    ),
    "schedule": (
        ATTACK_353 / "validation_inputs"
        / "validated_schedule_339.parquet"
    ),
    "gt_users": (
        ATTACK_353 / "data"
        / "ground_truth_users.parquet"
    ),
    "features": (
        ATTACK_353 / "data"
        / "behavior_features_injected_final.parquet"
    )
}

required_353 = [
    "user_day", "catalog", "schedule",
    "gt_users", "features"
]

for name in required_353:
    assert PATHS_353[name].is_file(), (
        f"Missing required file: {PATHS_353[name]}"
    )

if PATHS_353["baseline_parquet"].is_file():
    baseline_path_353 = PATHS_353["baseline_parquet"]
    baseline_353 = pd.read_parquet(baseline_path_353)
elif PATHS_353["baseline_csv"].is_file():
    baseline_path_353 = PATHS_353["baseline_csv"]
    baseline_353 = pd.read_csv(baseline_path_353)
else:
    raise FileNotFoundError(
        "Original User Baseline file not found."
    )

schedule_353 = pd.read_parquet(PATHS_353["schedule"])
catalog_353 = pd.read_parquet(PATHS_353["catalog"])
users_353 = pd.read_parquet(PATHS_353["gt_users"])
saved_353 = pd.read_parquet(PATHS_353["features"])

assert saved_353.shape == (69634, 115)
assert len(schedule_353) == 15010
assert len(users_353) == 670

print("\nA. SOURCE FILE DISCOVERY")
print("Original User-Day:", PATHS_353["user_day"])
print("Original Baseline:", baseline_path_353)
print("Approved Catalog:", PATHS_353["catalog"])
print("Schedule:", PATHS_353["schedule"])
print("Saved Features:", PATHS_353["features"])
print("Status: PASS")

# ------------------------------------------------------------
# 2. Reload complete original User-Day records.
#
# DuckDB filters the Parquet before loading it into Pandas.
# The feature computations below use Pandas independently.
# ------------------------------------------------------------

connection_353 = duckdb.connect(database=":memory:")

raw_353 = connection_353.execute(
    """
    SELECT d.*
    FROM read_parquet(?) AS d
    INNER JOIN (
        SELECT DISTINCT user
        FROM read_parquet(?)
    ) AS u
        ON d.user = u.user
    WHERE d.day >= DATE '2011-01-01'
      AND d.is_weekend = 0
      AND d.logon_count > 0
    ORDER BY d.user, d.day
    """,
    [
        str(PATHS_353["user_day"]),
        str(PATHS_353["gt_users"])
    ]
).df()

connection_353.close()

KEYS_353 = ["user", "day"]

for frame in [raw_353, schedule_353, saved_353]:
    frame["day"] = (
        pd.to_datetime(frame["day"], errors="raise")
        .astype("datetime64[ns]")
        .dt.normalize()
    )

raw_353 = raw_353.sort_values(
    KEYS_353
).reset_index(drop=True)

saved_353 = saved_353.sort_values(
    KEYS_353
).reset_index(drop=True)

assert raw_353.shape == (69634, 34)
assert raw_353["user"].nunique() == 670
assert not raw_353.duplicated(KEYS_353).any()

pd.testing.assert_frame_equal(
    raw_353[KEYS_353],
    saved_353[KEYS_353],
    check_exact=True
)

print("\nB. ORIGINAL WORKDAY SOURCE")
print("Users:", raw_353["user"].nunique())
print("User-days:", len(raw_353))
print("Original daily columns:", len(raw_353.columns))
print("User-day alignment: PASS")

# ------------------------------------------------------------
# 3. Reconstruct Formula B injection independently.
#
# No use of expected_351, selected_injection_3424,
# full_variants_3410, or any prior feature calculations.
# ------------------------------------------------------------

def parse_channels_353(value):
    if pd.isna(value):
        return []
    return [
        item.strip().lower()
        for item in str(value).split("|")
        if item.strip()
    ]

planned_353 = schedule_353[
    [
        "user", "day", "scenario_id",
        "planned_channels", "attack_channels"
    ]
].copy()

planned_353["channel_list"] = (
    planned_353["attack_channels"]
    .map(parse_channels_353)
)

assert (
    planned_353["channel_list"].map(len)
    == planned_353["planned_channels"]
).all()

planned_353 = (
    planned_353
    .explode("channel_list")
    .rename(columns={"channel_list": "channel"})
)

planned_353 = planned_353.loc[
    planned_353["channel"].notna(),
    ["user", "day", "scenario_id", "channel"]
].copy()

assert len(planned_353) == 14375
assert not planned_353.duplicated(
    ["user", "day", "channel"]
).any()

assert catalog_353["scenario_id"].is_unique

planned_353 = planned_353.merge(
    catalog_353[
        ["scenario_id", "stealth_level", "stealth_multiplier"]
    ],
    on="scenario_id",
    how="left",
    validate="many_to_one"
)

assert planned_353[
    ["stealth_level", "stealth_multiplier"]
].notna().all().all()

MULTIPLIERS_353 = {
    "High": 0.5,
    "Medium": 1.0,
    "Low": 1.5
}

np.testing.assert_allclose(
    planned_353["stealth_multiplier"],
    planned_353["stealth_level"].map(
        MULTIPLIERS_353
    ),
    atol=1e-12,
    rtol=0
)

SOURCE_COUNT_353 = {
    "email": "external_email_count",
    "usb": "usb_copy_count",
    "http": "www_upload_count"
}

planned_353 = planned_353.merge(
    raw_353[
        KEYS_353 + list(SOURCE_COUNT_353.values())
    ],
    on=KEYS_353,
    how="left",
    validate="many_to_one",
    indicator=True
)

assert planned_353["_merge"].eq("both").all()
planned_353.drop(columns="_merge", inplace=True)

planned_353["natural_count"] = np.nan

for channel, source_col in SOURCE_COUNT_353.items():
    mask = planned_353["channel"].eq(channel)
    planned_353.loc[mask, "natural_count"] = (
        planned_353.loc[mask, source_col]
        .to_numpy(dtype=float)
    )

assert planned_353["natural_count"].gt(0).all()

planned_353["independent_injection"] = np.maximum(
    1,
    np.ceil(
        planned_353["natural_count"].to_numpy(dtype=float)
        * planned_353["stealth_multiplier"].to_numpy(dtype=float)
    )
).astype(np.int64)

assert int(
    planned_353["independent_injection"].sum()
) == 42095

# Pivot the independently calculated additions.
deltas_353 = (
    planned_353
    .pivot(
        index=KEYS_353,
        columns="channel",
        values="independent_injection"
    )
    .reindex(
        columns=["email", "usb", "http"],
        fill_value=0
    )
    .fillna(0)
    .reset_index()
    .rename(columns={
        "email": "_delta_email_353",
        "usb": "_delta_usb_353",
        "http": "_delta_http_353"
    })
)

rebuilt_353 = raw_353.merge(
    deltas_353,
    on=KEYS_353,
    how="left",
    validate="one_to_one"
)

delta_cols_353 = [
    "_delta_email_353",
    "_delta_usb_353",
    "_delta_http_353"
]

rebuilt_353[delta_cols_353] = (
    rebuilt_353[delta_cols_353]
    .fillna(0)
    .astype(np.int64)
)

print("\nC. INDEPENDENT FORMULA B RECONSTRUCTION")
print("Planned channel-days:", len(planned_353))
print("Injected events:", int(
    planned_353["independent_injection"].sum()
))
print("Status: PASS")

# ------------------------------------------------------------
# 4. Independently reconstruct aggregate-count dependencies.
#
# Email:
# external + send + event
#
# USB:
# usb copy + file copy + file event
#
# HTTP:
# upload + http event
# ------------------------------------------------------------

COUNT_DEPENDENCIES_353 = {
    "_delta_email_353": [
        "external_email_count",
        "email_send_count",
        "email_event_count"
    ],
    "_delta_usb_353": [
        "usb_copy_count",
        "file_copy_count",
        "file_event_count"
    ],
    "_delta_http_353": [
        "www_upload_count",
        "http_event_count"
    ]
}

for delta_col, count_columns in COUNT_DEPENDENCIES_353.items():

    addition = rebuilt_353[delta_col].to_numpy(
        dtype=float
    )

    for count_col in count_columns:

        original_values = pd.to_numeric(
            rebuilt_353[count_col],
            errors="raise"
        ).to_numpy(dtype=float)

        rebuilt_353[count_col] = (
            original_values + addition
        )

rebuilt_353.drop(
    columns=delta_cols_353,
    inplace=True
)

# ------------------------------------------------------------
# 5. Validate aggregate event-count identities.
# ------------------------------------------------------------

COUNT_IDENTITIES_353 = {
    "Email Sends": (
        "email_send_count",
        ["external_email_count", "internal_email_count"]
    ),
    "Email Events": (
        "email_event_count",
        ["email_send_count", "email_view_count"]
    ),
    "File Events": (
        "file_event_count",
        [
            "file_copy_count", "file_open_count",
            "file_write_count", "file_delete_count"
        ]
    ),
    "HTTP Events": (
        "http_event_count",
        [
            "www_visit_count", "www_download_count",
            "www_upload_count"
        ]
    )
}

identity_rows_353 = []

for name, (total_col, part_cols) in (
    COUNT_IDENTITIES_353.items()
):

    expected_total = rebuilt_353[
        part_cols
    ].sum(axis=1).to_numpy(dtype=float)

    actual_total = rebuilt_353[
        total_col
    ].to_numpy(dtype=float)

    equal = np.isclose(
        actual_total,
        expected_total,
        rtol=0,
        atol=1e-9,
        equal_nan=False
    )

    identity_rows_353.append({
        "Identity": name,
        "Compared_Rows": len(equal),
        "Violations": int((~equal).sum())
    })

identity_audit_353 = pd.DataFrame(identity_rows_353)

print("\nD. AGGREGATE COUNT MATHEMATICAL IDENTITIES")
display(identity_audit_353)

# ------------------------------------------------------------
# 6. Join original historical baseline fields.
# ------------------------------------------------------------

BASELINE_COLUMNS_353 = [
    "baseline_workdays",
    "email_active_days",
    "email_active_rate",
    "email_count_p50",
    "email_count_p90",
    "email_count_p95",
    "email_count_mad",
    "email_bytes_p50",
    "email_bytes_p90",
    "email_bytes_p95",
    "email_bytes_mad",
    "usb_active_days",
    "usb_active_rate",
    "usb_copy_p50",
    "usb_copy_p90",
    "usb_copy_p95",
    "usb_copy_mad",
    "http_upload_active_days",
    "http_upload_active_rate",
    "http_upload_p50",
    "http_upload_p90",
    "http_upload_p95",
    "http_upload_mad"
]

assert len(BASELINE_COLUMNS_353) == 23
assert baseline_353["user"].is_unique

assert set(BASELINE_COLUMNS_353).issubset(
    baseline_353.columns
)

rebuilt_353 = rebuilt_353.merge(
    baseline_353[
        ["user"] + BASELINE_COLUMNS_353
    ],
    on="user",
    how="left",
    validate="many_to_one"
)

assert rebuilt_353["baseline_workdays"].notna().all()

# ------------------------------------------------------------
# 7. Independently calculate today's active channels.
# ------------------------------------------------------------

ACTIVE_SPECS_353 = {
    "email_active_today": "external_email_count",
    "usb_active_today": "usb_copy_count",
    "http_upload_active_today": "www_upload_count"
}

for output_col, source_col in ACTIVE_SPECS_353.items():
    rebuilt_353[output_col] = (
        rebuilt_353[source_col] > 0
    ).astype(np.int64)

rebuilt_353["active_channel_count"] = (
    rebuilt_353["email_active_today"]
    + rebuilt_353["usb_active_today"]
    + rebuilt_353["http_upload_active_today"]
)

assert rebuilt_353["active_channel_count"].between(
    0, 3
).all()

# ------------------------------------------------------------
# 8. Independently calculate P95 flags and ratios.
# ------------------------------------------------------------

P95_SPECS_353 = {
    "email_count": (
        "external_email_count", "email_count_p95"
    ),
    "email_bytes": (
        "external_email_bytes", "email_bytes_p95"
    ),
    "usb_copy": (
        "usb_copy_count", "usb_copy_p95"
    ),
    "http_upload": (
        "www_upload_count", "http_upload_p95"
    )
}

for prefix, (value_col, p95_col) in P95_SPECS_353.items():

    values = pd.to_numeric(
        rebuilt_353[value_col], errors="raise"
    )

    p95 = pd.to_numeric(
        rebuilt_353[p95_col], errors="raise"
    )

    # Original SQL: missing P95 does not count as above.
    rebuilt_353[f"{prefix}_above_p95"] = (
        p95.notna()
        & values.gt(p95)
    ).astype(np.int64)

    # Original SQL: ratio is null when P95 <= 0 or missing.
    rebuilt_353[f"{prefix}_to_p95_ratio"] = (
        values.div(p95.where(p95.gt(0)))
    )

# ------------------------------------------------------------
# 9. Recalculate the original causal rolling features.
#
# IMPORTANT:
# These original features use min_periods=1.
# That is DIFFERENT from the 30 new features.
# ------------------------------------------------------------

rebuilt_353 = (
    rebuilt_353
    .sort_values(KEYS_353)
    .reset_index(drop=True)
)

ACTIVE_ROLLING_353 = {
    "email_active_today": "email_active_days",
    "usb_active_today": "usb_active_days",
    "http_upload_active_today": "http_active_days"
}

for source_col, prefix in ACTIVE_ROLLING_353.items():

    for window in [5, 10, 20]:

        output_col = f"{prefix}_{window}wd"

        rebuilt_353[output_col] = (
            rebuilt_353.groupby(
                "user", sort=False
            )[source_col]
            .transform(
                lambda s, w=window: (
                    s.rolling(
                        window=w,
                        min_periods=1
                    ).sum()
                )
            )
        )

COUNT_ROLLING_353 = {
    "external_email_count": "external_email_count",
    "usb_copy_count": "usb_copy_count",
    "www_upload_count": "http_upload_count"
}

for source_col, prefix in COUNT_ROLLING_353.items():

    for window in [5, 10]:

        output_col = f"{prefix}_{window}wd"

        rebuilt_353[output_col] = (
            rebuilt_353.groupby(
                "user", sort=False
            )[source_col]
            .transform(
                lambda s, w=window: (
                    s.rolling(
                        window=w,
                        min_periods=1
                    ).sum()
                )
            )
        )

# Original multi-channel activity:
# At least two active outbound channels in one day.

rebuilt_353["_multi_today_353"] = (
    rebuilt_353["active_channel_count"] >= 2
).astype(np.int64)

rebuilt_353["multi_channel_days_10wd"] = (
    rebuilt_353.groupby(
        "user", sort=False
    )["_multi_today_353"]
    .transform(
        lambda s: s.rolling(
            window=10,
            min_periods=1
        ).sum()
    )
)

rebuilt_353.drop(
    columns=["_multi_today_353"],
    inplace=True
)

# ------------------------------------------------------------
# 10. Verify the complete original 85-column schema.
# ------------------------------------------------------------

assert len(rebuilt_353.columns) == 85

SAVED_85_COLUMNS_353 = list(
    saved_353.columns[:85]
)

assert SAVED_85_COLUMNS_353[:2] == ["user", "day"]

missing_columns_353 = (
    set(SAVED_85_COLUMNS_353) - set(rebuilt_353.columns)
)

unexpected_columns_353 = (
    set(rebuilt_353.columns) - set(SAVED_85_COLUMNS_353)
)

assert not missing_columns_353, (
    f"Missing reconstructed columns: {missing_columns_353}"
)

assert not unexpected_columns_353, (
    f"Unexpected reconstructed columns: {unexpected_columns_353}"
)

rebuilt_353 = rebuilt_353[
    SAVED_85_COLUMNS_353
].copy()

saved_original_353 = saved_353[
    SAVED_85_COLUMNS_353
].copy()

pd.testing.assert_frame_equal(
    rebuilt_353[KEYS_353],
    saved_original_353[KEYS_353],
    check_exact=True
)

print("\nE. ORIGINAL 85-COLUMN REBUILD")
print("Rows:", len(rebuilt_353))
print("Columns:", len(rebuilt_353.columns))
print("User-day keys: PASS")
print("Full schema membership: PASS")

# ------------------------------------------------------------
# 11. Compare EVERY original feature column, row by row.
#
# Null locations are checked explicitly.
# Numeric comparisons allow only tiny floating error.
# Timestamps and other values are compared exactly.
# ------------------------------------------------------------

comparison_rows_353 = []
mismatch_examples_353 = {}

for column in SAVED_85_COLUMNS_353[2:]:

    expected_series = rebuilt_353[column]
    saved_series = saved_original_353[column]

    expected_null = expected_series.isna().to_numpy()
    saved_null = saved_series.isna().to_numpy()

    null_mismatch = expected_null != saved_null

    both_valid = ~expected_null & ~saved_null

    equal = np.zeros(
        len(rebuilt_353),
        dtype=bool
    )

    both_null = expected_null & saved_null
    equal[both_null] = True

    max_abs_error = np.nan

    if (
        pd.api.types.is_numeric_dtype(expected_series)
        and pd.api.types.is_numeric_dtype(saved_series)
    ):

        left_values = expected_series.to_numpy(
            dtype=float,
            na_value=np.nan
        )

        right_values = saved_series.to_numpy(
            dtype=float,
            na_value=np.nan
        )

        equal[both_valid] = np.isclose(
            left_values[both_valid],
            right_values[both_valid],
            rtol=1e-12,
            atol=1e-9,
            equal_nan=False
        )

        if both_valid.any():
            max_abs_error = float(
                np.max(
                    np.abs(
                        left_values[both_valid]
                        - right_values[both_valid]
                    )
                )
            )

    elif (
        column in ["first_logon", "last_logoff"]
        or pd.api.types.is_datetime64_any_dtype(
            expected_series
        )
        or pd.api.types.is_datetime64_any_dtype(
            saved_series
        )
    ):

        left_dates = pd.to_datetime(
            expected_series,
            errors="raise"
        ).astype("datetime64[ns]")

        right_dates = pd.to_datetime(
            saved_series,
            errors="raise"
        ).astype("datetime64[ns]")

        equal[both_valid] = (
            left_dates[both_valid].to_numpy()
            == right_dates[both_valid].to_numpy()
        )

    else:

        equal[both_valid] = (
            expected_series[both_valid]
            .astype(str)
            .to_numpy()
            ==
            saved_series[both_valid]
            .astype(str)
            .to_numpy()
        )

    mismatched = ~equal

    mismatch_count = int(mismatched.sum())

    if column in raw_353.columns:
        category = "Original Daily Source"
    elif column in BASELINE_COLUMNS_353:
        category = "Historical Baseline"
    elif (
        column.endswith("_5wd")
        or column.endswith("_10wd")
        or column.endswith("_20wd")
    ):
        category = "Original Rolling Features"
    else:
        category = "Derived Daily Features"

    comparison_rows_353.append({
        "Column": column,
        "Category": category,
        "Compared_Rows": len(rebuilt_353),
        "Mismatches": mismatch_count,
        "Null_Mismatches": int(null_mismatch.sum()),
        "Max_Absolute_Error": max_abs_error
    })

    if mismatch_count > 0:

        example_positions = np.flatnonzero(
            mismatched
        )[:10]

        mismatch_examples_353[column] = pd.DataFrame({
            "user": rebuilt_353.iloc[
                example_positions
            ]["user"].to_numpy(),

            "day": rebuilt_353.iloc[
                example_positions
            ]["day"].to_numpy(),

            "Expected": expected_series.iloc[
                example_positions
            ].to_numpy(),

            "Saved": saved_series.iloc[
                example_positions
            ].to_numpy()
        })

feature_audit_353 = pd.DataFrame(
    comparison_rows_353
)

# ------------------------------------------------------------
# 12. Report results by feature category.
# ------------------------------------------------------------

print("\nF. FULL FEATURE VALIDATION BY CATEGORY")

category_summary_353 = (
    feature_audit_353
    .groupby("Category", as_index=False)
    .agg(
        Columns=("Column", "size"),
        Compared_Values=("Compared_Rows", "sum"),
        Mismatches=("Mismatches", "sum"),
        Null_Mismatches=("Null_Mismatches", "sum")
    )
)

display(category_summary_353)

print("\nG. COLUMN-BY-COLUMN AUDIT")

display(feature_audit_353)

if mismatch_examples_353:

    print("\nH. MISMATCH EXAMPLES")

    for column, examples in mismatch_examples_353.items():

        print(f"\nColumn: {column}")
        display(examples)

else:
    print("\nH. MISMATCH EXAMPLES")
    print("No value mismatches found.")

# ------------------------------------------------------------
# 13. Final independent validation decision.
# ------------------------------------------------------------

total_mismatches_353 = int(
    feature_audit_353["Mismatches"].sum()
)

total_null_mismatches_353 = int(
    feature_audit_353["Null_Mismatches"].sum()
)

identity_violations_353 = int(
    identity_audit_353["Violations"].sum()
)

compared_values_353 = int(
    feature_audit_353["Compared_Rows"].sum()
)

print("\n" + "=" * 80)
print("STEP 3.5.3 — INDEPENDENT FEATURE AUDIT RESULTS")
print("=" * 80)

print("Original source User-Day: RELOADED")
print("Historical baseline: RELOADED")
print("Scenario catalog and schedule: RELOADED")
print("Formula B: INDEPENDENTLY RECALCULATED")
print("Original 85-column structure: REBUILT")

print("Original columns:", len(rebuilt_353.columns))
print("Non-key columns checked:", len(feature_audit_353))
print("Feature values compared:", compared_values_353)
print("Feature value mismatches:", total_mismatches_353)
print("Null-placement mismatches:", total_null_mismatches_353)
print("Aggregate count identity violations:", identity_violations_353)

print("Files modified: NO")
print("Files exported: NO")

assert identity_violations_353 == 0, (
    "Aggregate event-count identities FAILED."
)

assert total_mismatches_353 == 0, (
    "Independent original feature validation FAILED. "
    "Review mismatch examples and categories."
)

assert len(feature_audit_353) == 83
assert compared_values_353 == 69634 * 83

print("\nSTEP 3.5.3 RESULT: PASS")
print("Original daily source fields: VERIFIED")
print("Historical baseline fields: VERIFIED")
print("Derived daily behavioral features: VERIFIED")
print("Original rolling-window features: VERIFIED")
print("Remaining Stage 3.5 checks: NOT YET PERFORMED")


STAGE 3.5 — STEP 3.5.3
INDEPENDENT ORIGINAL 85-COLUMN FEATURE VALIDATION

A. SOURCE FILE DISCOVERY
Original User-Day: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Data_Engineering\Data_engineering\data\processed\user_day_r62_v1.parquet
Original Baseline: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Data_Engineering\Data_engineering\data\features\user_baseline_r62_v1.parquet
Approved Catalog: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Attack_Simulation\data\approved_scenario_catalog_r62.parquet
Schedule: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Attack_Simulation\validation_inputs\validated_schedule_339.parquet
Saved Features: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Attack_Simulation\data\behavior_features_injected_final.parquet
Status: PASS

B. ORIGINAL WORKDAY SOURCE
Users: 670
User-days: 69634
Original daily columns: 34
User-day alignment: PASS

C. INDEPENDENT FORMULA B RECONSTRUCTION
Planned channel-days: 14375
Injected events: 42095
Status: PASS

D. AGGREGA

,Identity,Compared_Rows,Violations
0,Email Sends,69634,0
1,Email Events,69634,0
2,File Events,69634,0
3,HTTP Events,69634,0



E. ORIGINAL 85-COLUMN REBUILD
Rows: 69634
Columns: 85
User-day keys: PASS
Full schema membership: PASS

F. FULL FEATURE VALIDATION BY CATEGORY


,Category,Columns,Compared_Values,Mismatches,Null_Mismatches
0,Derived Daily Features,12,835608,0,0
1,Historical Baseline,23,1601582,0,0
2,Original Daily Source,32,2228288,0,0
3,Original Rolling Features,16,1114144,0,0



G. COLUMN-BY-COLUMN AUDIT


,Column,Category,Compared_Rows,Mismatches,Null_Mismatches,Max_Absolute_Error
0,day_of_week,Original Daily Source,69634,0,0,0.0
1,is_weekend,Original Daily Source,69634,0,0,0.0
2,email_event_count,Original Daily Source,69634,0,0,0.0
3,email_send_count,Original Daily Source,69634,0,0,0.0
4,email_view_count,Original Daily Source,69634,0,0,0.0
...,...,...,...,...,...,...
78,usb_copy_count_5wd,Original Rolling Features,69634,0,0,0.0
79,usb_copy_count_10wd,Original Rolling Features,69634,0,0,0.0
80,http_upload_count_5wd,Original Rolling Features,69634,0,0,0.0
81,http_upload_count_10wd,Original Rolling Features,69634,0,0,0.0



H. MISMATCH EXAMPLES
No value mismatches found.

STEP 3.5.3 — INDEPENDENT FEATURE AUDIT RESULTS
Original source User-Day: RELOADED
Historical baseline: RELOADED
Scenario catalog and schedule: RELOADED
Formula B: INDEPENDENTLY RECALCULATED
Original 85-column structure: REBUILT
Original columns: 85
Non-key columns checked: 83
Feature values compared: 5779622
Feature value mismatches: 0
Null-placement mismatches: 0
Aggregate count identity violations: 0
Files modified: NO
Files exported: NO

STEP 3.5.3 RESULT: PASS
Original daily source fields: VERIFIED
Historical baseline fields: VERIFIED
Derived daily behavioral features: VERIFIED
Original rolling-window features: VERIFIED
Remaining Stage 3.5 checks: NOT YET PERFORMED



### Step 3.5.4 — Independent Recalculation of All 30 New Behavioral Features

**Objective:** Independently recalculate all 30 newly engineered behavioral features and compare every value with the exported final 115-column feature dataset.

**Inputs:**
- Independently reconstructed User-Day data from Step 3.5.3
- Original historical User Baseline
- Exported final feature dataset (comparison target only)

**Feature Groups:**

1. Cumulative Deviation — 5/10/20/30 WD
2. Combined Channel Intensity — 5/10/20/30 WD
3. Dominant Channel Switches — 5/10/20/30 WD
4. Channel Distribution Shift (TVD) — 5/10/20/30 WD
5. Consecutive Elevated Days — 5/10/20/30 WD
6. Cross-Channel Deviation Days — 5/10/20/30 WD
7. Email/USB/HTTP Rolling Totals — 20/30 WD

**Independent Calculation Rules:**

- Use the original P50/P95 values for each user and channel.
- Reconstruct the historical channel distribution from baseline active rates, active-day means, and P95.
- Calculate rolling windows using only the current and previous eligible workdays.
- Reconstruct observed dominant channels without attack labels.
- Exclude inactive and tied days from dominant-channel switching sequences.
- Leave incomplete windows as NaN.
- Leave TVD undefined for complete windows with no observed activity.
- Never fill missing historical P95 values with artificial thresholds.

**Validation Requirements:**

1. Verify exactly 30 new feature columns.
2. Independently test mathematical calculations on known examples.
3. Compare all 2,089,020 expected feature positions.
4. Verify the positions of missing values.
5. Verify switching, consecutive-day, count, and TVD bounds.
6. Report mismatches for every feature separately.
7. Show actual and expected values for any mismatching records.
8. Preserve all previously exported files.

**Expected Outcome:** Zero unexplained mismatches across all 30 new features.

**Safety:** Strictly read-only. No source or exported files are modified.


In [6]:

# Stage 3 — Step 3.5.4: Independent New Feature Validation
# Goal: Independently recalculate all 30 new features
# and compare every value with saved final features.
#
# Requires successful Step 3.5.3 in this QA notebook.
#
# STRICT READ-ONLY:
# No source files or exported files modified.

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from IPython.display import display

print("=" * 80)
print("STAGE 3.5 — STEP 3.5.4")
print("INDEPENDENT VALIDATION OF ALL 30 NEW FEATURES")
print("=" * 80)

# ------------------------------------------------------------
# 1. Prepare independent inputs from Step 3.5.3.
#
# rebuilt_353 was reconstructed from original sources.
# saved_353 is used ONLY as the comparison target.
# ------------------------------------------------------------

required_354 = [
    "rebuilt_353",
    "saved_353",
    "baseline_353"
]

for name in required_354:
    if name not in globals():
        raise RuntimeError(
            f"Missing {name}. Run Step 3.5.3 first."
        )

source_354 = rebuilt_353.copy(deep=True)
saved_354 = saved_353.copy(deep=True)
baseline_354 = baseline_353.copy(deep=True)

KEYS_354 = ["user", "day"]
WINDOWS_354 = [5, 10, 20, 30]

for frame in [source_354, saved_354]:
    frame["day"] = (
        pd.to_datetime(frame["day"], errors="raise")
        .astype("datetime64[ns]")
        .dt.normalize()
    )

    frame.sort_values(
        KEYS_354,
        inplace=True
    )

    frame.reset_index(
        drop=True,
        inplace=True
    )

assert source_354.shape == (69634, 85)
assert saved_354.shape == (69634, 115)

pd.testing.assert_frame_equal(
    source_354[KEYS_354],
    saved_354[KEYS_354],
    check_exact=True
)

assert baseline_354["user"].is_unique

print("\nA. INPUT ALIGNMENT: PASS")

# ------------------------------------------------------------
# 2. Define every feature and its original input fields.
# ------------------------------------------------------------

CHANNELS_354 = {
    "email": {
        "count": "external_email_count",
        "p50": "email_count_p50",
        "p95": "email_count_p95",
        "rate": "email_active_rate",
        "active_days": "email_active_days",
        "mean_candidates": [
            "email_count_mean_active",
            "email_count_active_mean",
            "external_email_count_mean_active",
            "email_mean_active",
            "email_count_mean"
        ]
    },
    "usb": {
        "count": "usb_copy_count",
        "p50": "usb_copy_p50",
        "p95": "usb_copy_p95",
        "rate": "usb_active_rate",
        "active_days": "usb_active_days",
        "mean_candidates": [
            "usb_copy_mean_active",
            "usb_copy_active_mean",
            "usb_mean_active",
            "usb_copy_mean"
        ]
    },
    "http": {
        "count": "www_upload_count",
        "p50": "http_upload_p50",
        "p95": "http_upload_p95",
        "rate": "http_upload_active_rate",
        "active_days": "http_upload_active_days",
        "mean_candidates": [
            "http_upload_mean_active",
            "http_upload_active_mean",
            "www_upload_mean_active",
            "http_upload_mean"
        ]
    }
}

GROUP_PREFIXES_354 = [
    "cumulative_deviation",
    "combined_channel_intensity",
    "dominant_channel_switches",
    "channel_distribution_shift",
    "consecutive_elevated_days",
    "cross_channel_deviation_days"
]

FEATURE_NAMES_354 = [
    f"{prefix}_{window}wd"
    for prefix in GROUP_PREFIXES_354
    for window in WINDOWS_354
]

for window in [20, 30]:
    for prefix in [
        "external_email_count",
        "usb_copy_count",
        "http_upload_count"
    ]:
        FEATURE_NAMES_354.append(
            f"{prefix}_{window}wd"
        )

assert len(FEATURE_NAMES_354) == 30
assert len(set(FEATURE_NAMES_354)) == 30

saved_new_columns_354 = set(
    saved_354.columns[85:]
)

assert set(FEATURE_NAMES_354) == saved_new_columns_354, (
    "The exported 30-feature schema differs "
    "from the approved design."
)

# Resolve the historical active-day mean column
# from the original baseline schema.
#
# No invented values or guessed numeric fallbacks:
# stop if the required input cannot be identified.

resolved_means_354 = {}

for channel, spec in CHANNELS_354.items():

    matches = [
        col
        for col in spec["mean_candidates"]
        if col in baseline_354.columns
    ]

    if not matches:
        print(
            "\nAvailable baseline columns:",
            baseline_354.columns.tolist()
        )

        raise KeyError(
            f"No recognized active-day mean column "
            f"for {channel}. Inspect the baseline schema "
            "before continuing."
        )

    resolved_means_354[channel] = matches[0]

print("\nB. HISTORICAL MEAN INPUTS")

display(pd.DataFrame([
    {
        "Channel": channel.upper(),
        "Baseline_Mean_Column": column
    }
    for channel, column in resolved_means_354.items()
]))

# ------------------------------------------------------------
# 3. Independently construct channel-level daily arrays.
# ------------------------------------------------------------

n_354 = len(source_354)

baseline_lookup_354 = (
    baseline_354.set_index("user")
    .reindex(source_354["user"])
)

assert len(baseline_lookup_354) == n_354

counts_354 = {}
normalized_354 = {}
elevated_354 = {}

daily_deviation_354 = np.zeros(n_354, dtype=float)
daily_intensity_sum_354 = np.zeros(n_354, dtype=float)
valid_channels_354 = np.zeros(n_354, dtype=np.int64)
elevated_count_354 = np.zeros(n_354, dtype=np.int8)

historical_weights_354 = []

for channel, spec in CHANNELS_354.items():

    count = pd.to_numeric(
        source_354[spec["count"]],
        errors="raise"
    ).to_numpy(dtype=float)

    p50 = pd.to_numeric(
        source_354[spec["p50"]],
        errors="raise"
    ).to_numpy(dtype=float)

    p95 = pd.to_numeric(
        source_354[spec["p95"]],
        errors="raise"
    ).to_numpy(dtype=float)

    active_rate = pd.to_numeric(
        baseline_lookup_354[spec["rate"]],
        errors="raise"
    ).to_numpy(dtype=float)

    active_days = pd.to_numeric(
        baseline_lookup_354[spec["active_days"]],
        errors="raise"
    ).to_numpy(dtype=float)

    mean_active = pd.to_numeric(
        baseline_lookup_354[
            resolved_means_354[channel]
        ],
        errors="raise"
    ).to_numpy(dtype=float)

    assert np.isfinite(count).all()
    assert (count >= 0).all()

    valid_p95 = (
        np.isfinite(p95)
        & (p95 > 0)
    )

    valid_baseline = (
        valid_p95
        & np.isfinite(p50)
        & (p50 > 0)
        & (p95 >= p50)
    )

    assert not (
        (count > 0) & ~valid_baseline
    ).any(), (
        f"{channel}: positive count without "
        "a valid historical P50/P95 baseline."
    )

    # A missing baseline is acceptable only when
    # historical activity is verified as absent.
    invalid_history = ~valid_p95

    verified_inactive = (
        (active_days == 0)
        & (
            (active_rate == 0)
            | np.isnan(active_rate)
        )
    )

    assert not (
        invalid_history & ~verified_inactive
    ).any(), (
        f"{channel}: missing P95 without "
        "verified historical inactivity."
    )

    # ---------------------------------------------
    # 3.1 Normalized activity: Count / P95
    # ---------------------------------------------

    norm = np.zeros(n_354, dtype=float)

    np.divide(
        count,
        p95,
        out=norm,
        where=valid_p95
    )

    # ---------------------------------------------
    # 3.2 Positive deviation above P50
    # ---------------------------------------------

    deviation = np.zeros(n_354, dtype=float)

    numerator = np.maximum(
        0.0,
        count - p50
    )

    denominator = np.maximum(
        1.0,
        p95 - p50
    )

    np.divide(
        numerator,
        denominator,
        out=deviation,
        where=valid_baseline
    )

    # ---------------------------------------------
    # 3.3 Elevated channel indicator
    # ---------------------------------------------

    elevated = (
        valid_baseline
        & (count > p50)
    )

    # ---------------------------------------------
    # 3.4 Historical normalized channel weight
    #
    # Weight = Active Rate × Mean Active Count / P95
    # ---------------------------------------------

    weight = np.zeros(n_354, dtype=float)

    assert np.isfinite(
        active_rate[valid_p95]
    ).all()

    assert np.isfinite(
        mean_active[valid_p95]
    ).all()

    assert (
        active_rate[valid_p95] >= 0
    ).all()

    assert (
        mean_active[valid_p95] >= 0
    ).all()

    weight[valid_p95] = (
        active_rate[valid_p95]
        * mean_active[valid_p95]
        / p95[valid_p95]
    )

    counts_354[channel] = count
    normalized_354[channel] = norm
    elevated_354[channel] = elevated

    daily_deviation_354 += deviation
    daily_intensity_sum_354 += norm
    valid_channels_354 += valid_p95.astype(np.int64)
    elevated_count_354 += elevated.astype(np.int8)

    historical_weights_354.append(weight)

assert (valid_channels_354 > 0).all()

daily_intensity_354 = (
    daily_intensity_sum_354
    / valid_channels_354
)

cross_channel_354 = (
    elevated_count_354 >= 2
)

any_elevated_354 = (
    elevated_count_354 >= 1
)

# ------------------------------------------------------------
# 4. Reconstruct historical channel shares.
# ------------------------------------------------------------

historical_matrix_354 = np.column_stack(
    historical_weights_354
)

historical_total_354 = (
    historical_matrix_354.sum(axis=1)
)

assert np.isfinite(historical_matrix_354).all()
assert (historical_total_354 > 0).all()

historical_shares_354 = (
    historical_matrix_354
    / historical_total_354[:, None]
)

np.testing.assert_allclose(
    historical_shares_354.sum(axis=1),
    np.ones(n_354),
    rtol=0,
    atol=1e-10
)

normalized_matrix_354 = np.column_stack([
    normalized_354[channel]
    for channel in CHANNELS_354
])

# ------------------------------------------------------------
# 5. Independently identify the observed dominant channel.
#
# -1 = inactive day or tied dominant activity.
#  0 = Email, 1 = USB, 2 = HTTP.
#
# No attack labels or primary-channel schedule used.
# ------------------------------------------------------------

maximum_354 = normalized_matrix_354.max(axis=1)

tied_354 = np.isclose(
    normalized_matrix_354,
    maximum_354[:, None],
    rtol=1e-9,
    atol=1e-12
)

unique_positive_354 = (
    (maximum_354 > 0)
    & (tied_354.sum(axis=1) == 1)
)

dominant_354 = np.full(
    n_354,
    -1,
    dtype=np.int8
)

dominant_354[unique_positive_354] = (
    np.argmax(
        normalized_matrix_354,
        axis=1
    )[unique_positive_354]
)

print("\nC. DAILY BEHAVIOR DIAGNOSTICS")

daily_diagnostics_354 = pd.DataFrame([
    {
        "Measure": "Elevated in any channel",
        "User_Days": int(any_elevated_354.sum())
    },
    {
        "Measure": "Elevated in 2+ channels",
        "User_Days": int(cross_channel_354.sum())
    },
    {
        "Measure": "Unique positive dominant channel",
        "User_Days": int(unique_positive_354.sum())
    },
    {
        "Measure": "No positive channel activity",
        "User_Days": int((maximum_354 == 0).sum())
    },
    {
        "Measure": "Tied positive dominance",
        "User_Days": int(
            (
                (maximum_354 > 0)
                & ~unique_positive_354
            ).sum()
        )
    }
])

display(daily_diagnostics_354)

# ------------------------------------------------------------
# 6. Independent mathematical rolling helpers.
#
# Prefix-sum calculations are used for totals and means.
# Direct window inspection is used for switches and runs.
# ------------------------------------------------------------

def independent_window_sums_354(values, width):

    values = np.asarray(values, dtype=float)

    cumulative = np.concatenate([
        [0.0],
        np.cumsum(values, dtype=float)
    ])

    return (
        cumulative[width:]
        - cumulative[:-width]
    )


def independent_window_switches_354(labels, width):

    windows = sliding_window_view(
        np.asarray(labels),
        window_shape=width
    )

    result = np.zeros(
        len(windows),
        dtype=np.int64
    )

    for i, window_values in enumerate(windows):

        observed = window_values[
            window_values >= 0
        ]

        if len(observed) >= 2:

            result[i] = np.count_nonzero(
                observed[1:] != observed[:-1]
            )

    return result


def independent_window_runs_354(flags, width):

    windows = sliding_window_view(
        np.asarray(flags, dtype=bool),
        window_shape=width
    )

    current = np.zeros(
        len(windows),
        dtype=np.int64
    )

    longest = np.zeros(
        len(windows),
        dtype=np.int64
    )

    for j in range(width):

        current = np.where(
            windows[:, j],
            current + 1,
            0
        )

        longest = np.maximum(
            longest,
            current
        )

    return longest


# Known-answer self-tests.
np.testing.assert_array_equal(
    independent_window_switches_354(
        np.array([0, -1, 1, 1, 2]),
        5
    ),
    np.array([2])
)

np.testing.assert_array_equal(
    independent_window_runs_354(
        np.array([1, 1, 0, 1, 1, 1]),
        5
    ),
    np.array([2, 3])
)

np.testing.assert_allclose(
    independent_window_sums_354(
        np.array([1, 2, 3, 4, 5]),
        3
    ),
    np.array([6, 9, 12])
)

print("\nD. INDEPENDENT MATHEMATICAL SELF-TESTS: PASS")

# ------------------------------------------------------------
# 7. Independently recalculate all rolling features.
#
# Each user is processed independently.
# Results are NaN until W eligible days are available.
# ------------------------------------------------------------

independent_features_354 = pd.DataFrame(
    {
        "user": source_354["user"],
        "day": source_354["day"]
    }
)

for feature in FEATURE_NAMES_354:
    independent_features_354[feature] = np.nan

user_groups_354 = [
    np.asarray(indices, dtype=np.int64)
    for indices in source_354.groupby(
        "user",
        sort=False
    ).indices.values()
]

for indices in user_groups_354:

    length = len(indices)

    # Historical reference shares are fixed per user.
    historical_share = (
        historical_shares_354[indices[0]]
    )

    np.testing.assert_allclose(
        historical_shares_354[indices],
        np.broadcast_to(
            historical_share,
            (length, 3)
        ),
        rtol=0,
        atol=1e-10
    )

    for width in WINDOWS_354:

        if length < width:
            continue

        output_indices = indices[width - 1:]

        # ---------------------------------------------
        # 7.1 Cumulative deviation
        # ---------------------------------------------

        deviation_total = independent_window_sums_354(
            daily_deviation_354[indices],
            width
        )

        independent_features_354.loc[
            output_indices,
            f"cumulative_deviation_{width}wd"
        ] = deviation_total

        # ---------------------------------------------
        # 7.2 Combined channel intensity
        # ---------------------------------------------

        intensity_mean = (
            independent_window_sums_354(
                daily_intensity_354[indices],
                width
            ) / width
        )

        independent_features_354.loc[
            output_indices,
            f"combined_channel_intensity_{width}wd"
        ] = intensity_mean

        # ---------------------------------------------
        # 7.3 Observed dominant-channel switches
        # ---------------------------------------------

        switches = independent_window_switches_354(
            dominant_354[indices],
            width
        )

        independent_features_354.loc[
            output_indices,
            f"dominant_channel_switches_{width}wd"
        ] = switches

        # ---------------------------------------------
        # 7.4 Channel distribution shift (TVD)
        #
        # Observed shares are calculated from rolling
        # normalized activity totals.
        # ---------------------------------------------

        rolling_channels = np.column_stack([
            independent_window_sums_354(
                normalized_354[channel][indices],
                width
            )
            for channel in CHANNELS_354
        ])

        observed_totals = (
            rolling_channels.sum(axis=1)
        )

        has_activity = observed_totals > 0

        observed_shares = np.zeros_like(
            rolling_channels,
            dtype=float
        )

        np.divide(
            rolling_channels,
            observed_totals[:, None],
            out=observed_shares,
            where=has_activity[:, None]
        )

        tvd = (
            0.5
            * np.abs(
                observed_shares
                - historical_share[None, :]
            ).sum(axis=1)
        )

        # No distribution exists for a zero-activity window.
        tvd[~has_activity] = np.nan

        # Correct only floating-point boundary drift.
        tvd = np.clip(tvd, 0, 1)

        independent_features_354.loc[
            output_indices,
            f"channel_distribution_shift_{width}wd"
        ] = tvd

        # ---------------------------------------------
        # 7.5 Longest consecutive elevated-day run
        # ---------------------------------------------

        longest = independent_window_runs_354(
            any_elevated_354[indices],
            width
        )

        independent_features_354.loc[
            output_indices,
            f"consecutive_elevated_days_{width}wd"
        ] = longest

        # ---------------------------------------------
        # 7.6 Days with 2+ elevated channels
        # ---------------------------------------------

        cross_days = independent_window_sums_354(
            cross_channel_354[indices],
            width
        )

        independent_features_354.loc[
            output_indices,
            f"cross_channel_deviation_days_{width}wd"
        ] = cross_days

        # ---------------------------------------------
        # 7.7 Additional channel totals for 20/30 WD
        # ---------------------------------------------

        if width in [20, 30]:

            for channel, spec in CHANNELS_354.items():

                feature = (
                    f"{spec['count']}_{width}wd"
                )

                # HTTP feature uses the approved prefix
                # rather than the source column name.
                if channel == "http":
                    feature = f"http_upload_count_{width}wd"

                totals = independent_window_sums_354(
                    counts_354[channel][indices],
                    width
                )

                independent_features_354.loc[
                    output_indices,
                    feature
                ] = totals

assert len(independent_features_354) == 69634

assert len(independent_features_354.columns) == 32

print("\nE. ALL 30 FEATURES INDEPENDENTLY RECALCULATED")
print("Users:", len(user_groups_354))
print("Rows:", len(independent_features_354))
print("New features:", len(FEATURE_NAMES_354))

# ------------------------------------------------------------
# 8. Validate mathematical bounds and missing windows.
# ------------------------------------------------------------

print("\nF. MATHEMATICAL AND WINDOW-COVERAGE VALIDATION")

position_354 = (
    source_354.groupby("user").cumcount()
    .to_numpy()
    + 1
)

window_rows_354 = []
bounds_violations_354 = 0

for width in WINDOWS_354:

    complete = position_354 >= width

    representative = (
        f"cumulative_deviation_{width}wd"
    )

    actual_complete = (
        independent_features_354[
            representative
        ].notna().to_numpy()
    )

    assert np.array_equal(
        actual_complete,
        complete
    )

    for prefix, upper_bound in [
        ("dominant_channel_switches", width - 1),
        ("consecutive_elevated_days", width),
        ("cross_channel_deviation_days", width)
    ]:

        values = independent_features_354[
            f"{prefix}_{width}wd"
        ].to_numpy(dtype=float)

        valid_values = values[
            np.isfinite(values)
        ]

        violations = int(
            (
                (valid_values < 0)
                | (valid_values > upper_bound)
            ).sum()
        )

        bounds_violations_354 += violations

    tvd_values = independent_features_354[
        f"channel_distribution_shift_{width}wd"
    ].to_numpy(dtype=float)

    finite_tvd = tvd_values[np.isfinite(tvd_values)]

    bounds_violations_354 += int(
        (
            (finite_tvd < 0)
            | (finite_tvd > 1)
        ).sum()
    )

    window_rows_354.append({
        "Window_WD": width,
        "Complete_Windows": int(complete.sum()),
        "Incomplete_Windows": int((~complete).sum()),
        "TVD_Undefined_Complete_Windows": int(
            (
                complete
                & np.isnan(tvd_values)
            ).sum()
        )
    })

coverage_354 = pd.DataFrame(
    window_rows_354
)

display(coverage_354)

assert bounds_violations_354 == 0

# ------------------------------------------------------------
# 9. Compare EVERY reconstructed new feature value
#    against the saved final Parquet dataset.
#
# Check NaN placement separately from numeric equality.
# ------------------------------------------------------------

print("\nG. INDEPENDENT VALUE-BY-VALUE FEATURE AUDIT")

comparison_rows_354 = []
mismatch_examples_354 = {}

for feature in FEATURE_NAMES_354:

    expected = independent_features_354[
        feature
    ].to_numpy(dtype=float)

    saved_values = saved_354[
        feature
    ].to_numpy(dtype=float)

    expected_missing = np.isnan(expected)
    saved_missing = np.isnan(saved_values)

    null_mismatch = (
        expected_missing != saved_missing
    )

    equal = np.isclose(
        expected,
        saved_values,
        rtol=1e-11,
        atol=1e-9,
        equal_nan=True
    )

    differences = ~equal

    both_valid = (
        np.isfinite(expected)
        & np.isfinite(saved_values)
    )

    if both_valid.any():

        max_error = float(
            np.max(
                np.abs(
                    expected[both_valid]
                    - saved_values[both_valid]
                )
            )
        )

    else:
        max_error = np.nan

    mismatch_count = int(differences.sum())

    group = feature.rsplit("_", 1)[0]

    comparison_rows_354.append({
        "Feature": feature,
        "Group": group,
        "Compared_Rows": len(expected),
        "Value_Mismatches": mismatch_count,
        "Null_Mismatches": int(null_mismatch.sum()),
        "Max_Absolute_Error": max_error
    })

    if mismatch_count:

        positions = np.flatnonzero(
            differences
        )[:10]

        mismatch_examples_354[feature] = pd.DataFrame({
            "user": source_354.iloc[
                positions
            ]["user"].to_numpy(),
            "day": source_354.iloc[
                positions
            ]["day"].to_numpy(),
            "Expected": expected[positions],
            "Saved": saved_values[positions]
        })

feature_audit_354 = pd.DataFrame(
    comparison_rows_354
)

display(feature_audit_354)

# ------------------------------------------------------------
# 10. Summary of matched values and any discrepancies.
# ------------------------------------------------------------

print("\nH. NEW FEATURE VALIDATION SUMMARY")

total_compared_354 = int(
    feature_audit_354["Compared_Rows"].sum()
)

total_mismatches_354 = int(
    feature_audit_354["Value_Mismatches"].sum()
)

total_null_mismatches_354 = int(
    feature_audit_354["Null_Mismatches"].sum()
)

group_summary_354 = (
    feature_audit_354
    .groupby("Group", as_index=False)
    .agg(
        Features=("Feature", "size"),
        Compared_Values=("Compared_Rows", "sum"),
        Mismatches=("Value_Mismatches", "sum"),
        Null_Mismatches=("Null_Mismatches", "sum")
    )
)

display(group_summary_354)

if mismatch_examples_354:

    print("\nI. MISMATCH EXAMPLES")

    for feature, examples in mismatch_examples_354.items():

        print(f"\n{feature}")
        display(examples)

else:
    print("\nI. MISMATCH EXAMPLES")
    print("No mismatches found.")

# ------------------------------------------------------------
# 11. Final independent validation decision.
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STAGE 3.5 — STEP 3.5.4: FINAL AUDIT RESULTS")
print("=" * 80)

print("New features reconstructed:", len(FEATURE_NAMES_354))
print("User-days:", n_354)
print("Compared values:", total_compared_354)
print("Feature value mismatches:", total_mismatches_354)
print("Null-placement mismatches:", total_null_mismatches_354)
print("Mathematical bounds violations:", bounds_violations_354)
print("Input sources: ORIGINAL BASELINE + INDEPENDENT USER-DAY")
print("Saved feature data used only for comparison: YES")
print("Files modified: NO")
print("Files exported: NO")

assert total_compared_354 == 69634 * 30

assert total_mismatches_354 == 0, (
    "Independent new-feature validation FAILED. "
    "Review the mismatch tables."
)

assert total_null_mismatches_354 == 0

assert bounds_violations_354 == 0

print("\nSTEP 3.5.4 RESULT: PASS")
print("All 30 new behavioral features: VERIFIED")
print("Rolling windows and missing values: VERIFIED")
print("Historical channel distribution: VERIFIED")
print("Observed channel switches: VERIFIED")
print("Original and final files: UNCHANGED")
print("Remaining Stage 3.5 checks: NOT YET PERFORMED")


STAGE 3.5 — STEP 3.5.4
INDEPENDENT VALIDATION OF ALL 30 NEW FEATURES

A. INPUT ALIGNMENT: PASS

B. HISTORICAL MEAN INPUTS


,Channel,Baseline_Mean_Column
0,EMAIL,email_count_mean_active
1,USB,usb_copy_mean_active
2,HTTP,http_upload_mean_active



C. DAILY BEHAVIOR DIAGNOSTICS


,Measure,User_Days
0,Elevated in any channel,31514
1,Elevated in 2+ channels,4758
2,Unique positive dominant channel,56855
3,No positive channel activity,11650
4,Tied positive dominance,1129



D. INDEPENDENT MATHEMATICAL SELF-TESTS: PASS

E. ALL 30 FEATURES INDEPENDENTLY RECALCULATED
Users: 670
Rows: 69634
New features: 30

F. MATHEMATICAL AND WINDOW-COVERAGE VALIDATION


,Window_WD,Complete_Windows,Incomplete_Windows,TVD_Undefined_Complete_Windows
0,5,66954,2680,101
1,10,63606,6028,4
2,20,56920,12714,0
3,30,50241,19393,0



G. INDEPENDENT VALUE-BY-VALUE FEATURE AUDIT


,Feature,Group,Compared_Rows,Value_Mismatches,Null_Mismatches,Max_Absolute_Error
0,cumulative_deviation_5wd,cumulative_deviation,69634,0,0,2.664535e-14
1,cumulative_deviation_10wd,cumulative_deviation,69634,0,0,3.552714e-14
2,cumulative_deviation_20wd,cumulative_deviation,69634,0,0,4.973799e-14
3,cumulative_deviation_30wd,cumulative_deviation,69634,0,0,6.394885e-14
4,combined_channel_intensity_5wd,combined_channel_intensity,69634,0,0,2.942091e-15
5,combined_channel_intensity_10wd,combined_channel_intensity,69634,0,0,2.386980e-15
6,combined_channel_intensity_20wd,combined_channel_intensity,69634,0,0,1.554312e-15
7,combined_channel_intensity_30wd,combined_channel_intensity,69634,0,0,1.443290e-15
8,dominant_channel_switches_5wd,dominant_channel_switches,69634,0,0,0.000000e+00
9,dominant_channel_switches_10wd,dominant_channel_switches,69634,0,0,0.000000e+00



H. NEW FEATURE VALIDATION SUMMARY


,Group,Features,Compared_Values,Mismatches,Null_Mismatches
0,channel_distribution_shift,4,278536,0,0
1,combined_channel_intensity,4,278536,0,0
2,consecutive_elevated_days,4,278536,0,0
3,cross_channel_deviation_days,4,278536,0,0
4,cumulative_deviation,4,278536,0,0
5,dominant_channel_switches,4,278536,0,0
6,external_email_count,2,139268,0,0
7,http_upload_count,2,139268,0,0
8,usb_copy_count,2,139268,0,0



I. MISMATCH EXAMPLES
No mismatches found.

STAGE 3.5 — STEP 3.5.4: FINAL AUDIT RESULTS
New features reconstructed: 30
User-days: 69634
Compared values: 2089020
Feature value mismatches: 0
Null-placement mismatches: 0
Mathematical bounds violations: 0
Input sources: ORIGINAL BASELINE + INDEPENDENT USER-DAY
Saved feature data used only for comparison: YES
Files modified: NO
Files exported: NO

STEP 3.5.4 RESULT: PASS
All 30 new behavioral features: VERIFIED
Rolling windows and missing values: VERIFIED
Historical channel distribution: VERIFIED
Observed channel switches: VERIFIED
Original and final files: UNCHANGED
Remaining Stage 3.5 checks: NOT YET PERFORMED



### Step 3.5.5 — Independent Ground Truth Reconstruction & Audit

**Objective:** Independently reconstruct all three final Ground Truth tables and compare every record and non-key value with the exported files.

**Independent Sources:**
- Original CERT r6.2 User-Day dataset
- Original User Baseline
- Approved Scenario Catalog
- Validated Schedule reference

**Outputs Under Test:**
1. `ground_truth_users.parquet`
2. `ground_truth_channel_days.parquet`
3. `ground_truth_user_days.parquet`

**Independent Reconstruction:**

**User Level**
- Derive scenario assignments directly from the validated schedule.
- Derive stealth levels and scenario switching targets from the approved catalog.
- Recalculate window boundaries, duration, attack days, Off days, primary-channel switches, and total injected events.

**Channel-Day Level**
- Independently reconstruct selected channels from the schedule.
- Retrieve original daily counts and historical P95.
- Recalculate Formula B additions and final counts.
- Compare every user-day-channel record.

**User-Day Level**
- Independently identify all eligible workdays.
- Identify scheduled and unscheduled days.
- Determine Attack and Off labels from the schedule and independently calculated injections.
- Recalculate per-channel and total daily injected events.

**Validation Requirements:**
- Verify exact primary-key sets and uniqueness.
- Verify schema completeness and column values.
- Check missing-value placement and date values.
- Verify cross-table mathematical relationships.
- Verify scenario/stealth consistency.
- Check Attack, Off, and Outside Schedule classification.
- Confirm 42,095 injected events and absence of off-plan injections.
- Confirm Ground Truth columns are not present in the ML feature dataset.
- Report mismatch counts and examples for each column.

**Important:** An Attack label means a controlled synthetic injection occurred. An Outside Schedule or Off label means no synthetic injection occurred; it does not establish that the underlying activity is always benign.

**Safety:** Strictly read-only. No original or exported datasets are modified.

**Expected Outcome:** Independent validation of the complete three-table Ground Truth structure with zero unexplained mismatches.


In [7]:

# Stage 3 — Step 3.5.5: Independent Ground Truth Audit
# Goal: Independently reconstruct all three GT tables
# and compare every saved row and field.
#
# STRICT READ-ONLY:
# No source files or final outputs modified.

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 80)
print("STAGE 3.5 — STEP 3.5.5")
print("INDEPENDENT GROUND TRUTH RECONSTRUCTION & AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# 1. Resolve source files independently.
# ------------------------------------------------------------

ROOT_355 = Path(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration"
)

DE_355 = ROOT_355 / "Data_Engineering" / "Data_engineering"
ATTACK_355 = ROOT_355 / "Attack_Simulation"

PATHS_355 = {
    "raw": DE_355 / "data" / "processed"
           / "user_day_r62_v1.parquet",

    "baseline": DE_355 / "data" / "features"
                / "user_baseline_r62_v1.parquet",

    "catalog": ATTACK_355 / "data"
               / "approved_scenario_catalog_r62.parquet",

    "schedule": ATTACK_355 / "validation_inputs"
                / "validated_schedule_339.parquet",

    "gt_users": ATTACK_355 / "data"
                / "ground_truth_users.parquet",

    "gt_channels": ATTACK_355 / "data"
                   / "ground_truth_channel_days.parquet",

    "gt_days": ATTACK_355 / "data"
               / "ground_truth_user_days.parquet",

    "features": ATTACK_355 / "data"
                / "behavior_features_injected_final.parquet"
}

for name, path in PATHS_355.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {name}: {path}"
        )

# ------------------------------------------------------------
# 2. Reload original sources and saved targets.
# ------------------------------------------------------------

raw_355 = pd.read_parquet(
    PATHS_355["raw"],
    columns=[
        "user", "day", "is_weekend", "logon_count",
        "external_email_count",
        "usb_copy_count",
        "www_upload_count"
    ]
)

baseline_355 = pd.read_parquet(
    PATHS_355["baseline"],
    columns=[
        "user",
        "email_count_p95",
        "usb_copy_p95",
        "http_upload_p95"
    ]
)

catalog_355 = pd.read_parquet(
    PATHS_355["catalog"],
    columns=[
        "scenario_id",
        "stealth_level",
        "stealth_multiplier",
        "target_switches"
    ]
)

schedule_355 = pd.read_parquet(
    PATHS_355["schedule"]
)[
    [
        "user", "day", "scenario_id",
        "window_start", "window_end",
        "planned_channels", "attack_channels",
        "primary_channel"
    ]
].copy()

saved_users_355 = pd.read_parquet(PATHS_355["gt_users"])
saved_channels_355 = pd.read_parquet(PATHS_355["gt_channels"])
saved_days_355 = pd.read_parquet(PATHS_355["gt_days"])
saved_features_355 = pd.read_parquet(PATHS_355["features"])

def normalize_date_355(series):
    return (
        pd.to_datetime(series, errors="raise")
        .astype("datetime64[ns]")
        .dt.normalize()
    )

for frame in [
    raw_355, schedule_355,
    saved_channels_355, saved_days_355,
    saved_features_355
]:
    frame["day"] = normalize_date_355(frame["day"])

for col in ["window_start", "window_end"]:
    schedule_355[col] = normalize_date_355(
        schedule_355[col]
    )

for col in ["attack_start", "attack_end"]:
    saved_users_355[col] = normalize_date_355(
        saved_users_355[col]
    )

KEYS_355 = ["user", "day"]
CHANNEL_KEYS_355 = ["user", "day", "channel"]

assert len(schedule_355) == 15010
assert schedule_355["user"].nunique() == 670
assert not schedule_355.duplicated(KEYS_355).any()
assert baseline_355["user"].is_unique
assert catalog_355["scenario_id"].is_unique

print("\nA. SOURCE DISCOVERY AND LOADING: PASS")

# ------------------------------------------------------------
# 3. Rebuild the eligible user-day universe from raw data.
#
# Do not derive the population from saved GT users.
# ------------------------------------------------------------

simulated_users_355 = set(schedule_355["user"])

eligible_355 = raw_355.loc[
    raw_355["user"].isin(simulated_users_355)
    & raw_355["day"].ge(pd.Timestamp("2011-01-01"))
    & raw_355["is_weekend"].eq(0)
    & raw_355["logon_count"].gt(0)
].copy()

eligible_355 = (
    eligible_355.sort_values(KEYS_355)
    .reset_index(drop=True)
)

assert len(eligible_355) == 69634
assert eligible_355["user"].nunique() == 670
assert not eligible_355.duplicated(KEYS_355).any()

print("\nB. INDEPENDENT POPULATION")
print("Users:", len(simulated_users_355))
print("Eligible user-days:", len(eligible_355))

# ------------------------------------------------------------
# 4. Independently reconstruct scenario and user metadata.
# ------------------------------------------------------------

assert schedule_355.groupby("user")[
    "scenario_id"
].nunique().eq(1).all()

assert schedule_355.groupby("user")[
    "window_start"
].nunique().eq(1).all()

assert schedule_355.groupby("user")[
    "window_end"
].nunique().eq(1).all()

schedule_355 = (
    schedule_355
    .sort_values(KEYS_355)
    .reset_index(drop=True)
)

user_base_355 = (
    schedule_355.groupby("user", as_index=False)
    .agg(
        scenario_id=("scenario_id", "first"),
        attack_start=("window_start", "first"),
        attack_end=("window_end", "first"),
        duration_days=("day", "size"),
        attack_days=(
            "planned_channels",
            lambda x: int(x.gt(0).sum())
        ),
        off_days=(
            "planned_channels",
            lambda x: int(x.eq(0).sum())
        )
    )
)

user_base_355 = user_base_355.merge(
    catalog_355,
    on="scenario_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

assert user_base_355["_merge"].eq("both").all()
user_base_355.drop(columns="_merge", inplace=True)

# Independently recount primary-channel changes.
# Off days are skipped, not treated as switches.

switch_rows_355 = []
invalid_primary_355 = 0

for user, group in schedule_355.groupby("user"):

    previous = None
    switches = 0

    for row in group.itertuples(index=False):

        if int(row.planned_channels) == 0:
            continue

        selected = [
            x.strip().lower()
            for x in str(row.attack_channels).split("|")
            if x.strip()
        ]

        primary = str(row.primary_channel).strip().lower()

        if primary not in selected:
            invalid_primary_355 += 1
            continue

        if previous is not None and primary != previous:
            switches += 1

        previous = primary

    switch_rows_355.append({
        "user": user,
        "recalculated_switches": switches
    })

switch_df_355 = pd.DataFrame(switch_rows_355)

user_base_355 = user_base_355.merge(
    switch_df_355,
    on="user",
    validate="one_to_one"
)

assert invalid_primary_355 == 0

assert (
    user_base_355["recalculated_switches"]
    == user_base_355["target_switches"]
).all()

assert user_base_355["attack_days"].sum() == 12636
assert user_base_355["off_days"].sum() == 2374

# ------------------------------------------------------------
# 5. Expand scheduled attack channels independently.
# ------------------------------------------------------------

def parse_channels_355(value):
    if pd.isna(value):
        return []
    return [
        x.strip().lower()
        for x in str(value).split("|")
        if x.strip()
    ]

planned_355 = schedule_355[
    KEYS_355 + [
        "scenario_id",
        "planned_channels",
        "attack_channels"
    ]
].copy()

planned_355["channel_list"] = (
    planned_355["attack_channels"]
    .map(parse_channels_355)
)

assert (
    planned_355["channel_list"].map(len)
    == planned_355["planned_channels"]
).all()

assert planned_355["channel_list"].map(
    lambda x: len(x) == len(set(x))
).all()

planned_355 = (
    planned_355.explode("channel_list")
    .rename(columns={"channel_list": "channel"})
)

planned_355 = planned_355.loc[
    planned_355["channel"].notna(),
    KEYS_355 + ["scenario_id", "channel"]
].copy()

assert len(planned_355) == 14375

assert not planned_355.duplicated(
    CHANNEL_KEYS_355
).any()

assert set(planned_355["channel"]) == {
    "email", "usb", "http"
}

# ------------------------------------------------------------
# 6. Recalculate original counts, P95, and Formula B.
# ------------------------------------------------------------

planned_355 = planned_355.merge(
    eligible_355[
        KEYS_355 + [
            "external_email_count",
            "usb_copy_count",
            "www_upload_count"
        ]
    ],
    on=KEYS_355,
    how="left",
    validate="many_to_one",
    indicator=True
)

assert planned_355["_merge"].eq("both").all()
planned_355.drop(columns="_merge", inplace=True)

planned_355 = planned_355.merge(
    baseline_355,
    on="user",
    how="left",
    validate="many_to_one"
)

planned_355 = planned_355.merge(
    catalog_355[
        ["scenario_id", "stealth_level", "stealth_multiplier"]
    ],
    on="scenario_id",
    how="left",
    validate="many_to_one"
)

COUNT_COLS_355 = {
    "email": "external_email_count",
    "usb": "usb_copy_count",
    "http": "www_upload_count"
}

P95_COLS_355 = {
    "email": "email_count_p95",
    "usb": "usb_copy_p95",
    "http": "http_upload_p95"
}

planned_355["natural_count"] = np.nan
planned_355["historical_p95"] = np.nan

for channel in COUNT_COLS_355:

    mask = planned_355["channel"].eq(channel)

    planned_355.loc[
        mask, "natural_count"
    ] = planned_355.loc[
        mask, COUNT_COLS_355[channel]
    ].to_numpy(dtype=float)

    planned_355.loc[
        mask, "historical_p95"
    ] = planned_355.loc[
        mask, P95_COLS_355[channel]
    ].to_numpy(dtype=float)

assert planned_355["natural_count"].gt(0).all()
assert planned_355["historical_p95"].gt(0).all()

assert planned_355[
    ["stealth_level", "stealth_multiplier"]
].notna().all().all()

MULTIPLIERS_355 = {
    "High": 0.5,
    "Medium": 1.0,
    "Low": 1.5
}

np.testing.assert_allclose(
    planned_355["stealth_multiplier"],
    planned_355["stealth_level"].map(
        MULTIPLIERS_355
    ),
    rtol=0,
    atol=1e-12
)

planned_355["injected_count"] = np.maximum(
    1,
    np.ceil(
        planned_355["natural_count"].to_numpy(dtype=float)
        * planned_355["stealth_multiplier"].to_numpy(
            dtype=float
        )
    )
).astype(np.int64)

planned_355["final_count"] = (
    planned_355["natural_count"]
    + planned_355["injected_count"]
)

assert int(planned_355["injected_count"].sum()) == 42095

# ------------------------------------------------------------
# 7. Reconstruct Channel-Day Ground Truth.
# ------------------------------------------------------------

expected_channels_355 = planned_355[
    [
        "user", "day", "channel",
        "scenario_id", "stealth_level",
        "natural_count", "historical_p95",
        "stealth_multiplier", "injected_count",
        "final_count"
    ]
].copy()

expected_channels_355["injection_formula"] = (
    "B_Natural_Addition"
)

expected_channels_355 = (
    expected_channels_355
    .sort_values(CHANNEL_KEYS_355)
    .reset_index(drop=True)
)

assert expected_channels_355.shape == (14375, 11)

# ------------------------------------------------------------
# 8. Reconstruct User-Level Ground Truth.
# ------------------------------------------------------------

user_injection_totals_355 = (
    expected_channels_355.groupby(
        "user", as_index=False
    )
    .agg(
        injected_events_total=("injected_count", "sum")
    )
)

expected_users_355 = user_base_355.merge(
    user_injection_totals_355,
    on="user",
    how="left",
    validate="one_to_one"
)

expected_users_355["planned_switches"] = (
    expected_users_355["recalculated_switches"]
)

expected_users_355["injection_formula"] = (
    "B_Natural_Addition"
)

expected_users_355["ground_truth_source"] = (
    "synthetic_injection"
)

expected_users_355 = expected_users_355[
    [
        "user", "scenario_id", "stealth_level",
        "attack_start", "attack_end",
        "duration_days", "attack_days", "off_days",
        "planned_switches", "injected_events_total",
        "injection_formula", "ground_truth_source"
    ]
].sort_values("user").reset_index(drop=True)

assert expected_users_355.shape == (670, 12)

# ------------------------------------------------------------
# 9. Reconstruct daily channel injection totals.
# ------------------------------------------------------------

daily_counts_355 = (
    expected_channels_355
    .groupby(KEYS_355, as_index=False)
    .agg(
        injected_channel_count=("channel", "nunique"),
        injected_events_total=("injected_count", "sum")
    )
)

daily_channel_pivot_355 = (
    expected_channels_355
    .pivot_table(
        index=KEYS_355,
        columns="channel",
        values="injected_count",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(columns=["email", "usb", "http"], fill_value=0)
    .reset_index()
    .rename(columns={
        "email": "injected_email",
        "usb": "injected_usb",
        "http": "injected_http"
    })
)

daily_counts_355 = daily_counts_355.merge(
    daily_channel_pivot_355,
    on=KEYS_355,
    validate="one_to_one"
)

assert len(daily_counts_355) == 12636

# ------------------------------------------------------------
# 10. Reconstruct Full User-Day Ground Truth.
# ------------------------------------------------------------

expected_days_355 = eligible_355[
    KEYS_355
].merge(
    expected_users_355[
        ["user", "scenario_id", "stealth_level"]
    ],
    on="user",
    how="left",
    validate="many_to_one"
)

expected_days_355 = expected_days_355.merge(
    schedule_355[
        KEYS_355 + ["planned_channels"]
    ],
    on=KEYS_355,
    how="left",
    validate="one_to_one",
    indicator="_schedule_presence"
)

expected_days_355["is_scheduled"] = (
    expected_days_355["_schedule_presence"]
    .eq("both").astype(np.int8)
)

expected_days_355.drop(
    columns="_schedule_presence",
    inplace=True
)

expected_days_355 = expected_days_355.merge(
    daily_counts_355,
    on=KEYS_355,
    how="left",
    validate="one_to_one"
)

NUMERIC_COLS_355 = [
    "planned_channels",
    "injected_channel_count",
    "injected_email",
    "injected_usb",
    "injected_http",
    "injected_events_total"
]

expected_days_355[NUMERIC_COLS_355] = (
    expected_days_355[NUMERIC_COLS_355]
    .fillna(0)
    .astype(np.int64)
)

expected_days_355["is_attack_day"] = (
    expected_days_355["injected_events_total"]
    .gt(0).astype(np.int8)
)

expected_days_355["day_status"] = np.select(
    [
        expected_days_355["is_attack_day"].eq(1),
        expected_days_355["is_scheduled"].eq(1)
    ],
    ["Attack", "Off"],
    default="Outside Schedule"
)

expected_days_355 = expected_days_355[
    [
        "user", "day", "scenario_id", "stealth_level",
        "day_status", "is_scheduled", "is_attack_day",
        "planned_channels", "injected_channel_count",
        "injected_email", "injected_usb", "injected_http",
        "injected_events_total"
    ]
].sort_values(KEYS_355).reset_index(drop=True)

assert expected_days_355.shape == (69634, 13)

print("\nC. THREE GROUND TRUTH TABLES RECONSTRUCTED")

inventory_355 = pd.DataFrame([
    {
        "Dataset": "User-Level",
        "Rows": len(expected_users_355),
        "Columns": len(expected_users_355.columns)
    },
    {
        "Dataset": "Channel-Day",
        "Rows": len(expected_channels_355),
        "Columns": len(expected_channels_355.columns)
    },
    {
        "Dataset": "Full User-Day",
        "Rows": len(expected_days_355),
        "Columns": len(expected_days_355.columns)
    }
])

display(inventory_355)

# ------------------------------------------------------------
# 11. Generic independent table comparison.
#
# Compares:
# - Missing/unexpected keys
# - Each non-key value
# - Null placement
# - Numerical values with strict tolerance
# - Dates at nanosecond precision
# ------------------------------------------------------------

def audit_table_355(name, expected, saved, keys):

    expected = expected.copy(deep=True)
    saved = saved.copy(deep=True)

    assert not expected.duplicated(keys).any(), (
        f"{name}: duplicate expected keys"
    )

    assert not saved.duplicated(keys).any(), (
        f"{name}: duplicate saved keys"
    )

    missing_columns = set(expected.columns) - set(saved.columns)
    extra_columns = set(saved.columns) - set(expected.columns)

    if missing_columns or extra_columns:
        raise ValueError(
            f"{name} schema mismatch. "
            f"Missing: {missing_columns}; Extra: {extra_columns}"
        )

    comparison = expected.merge(
        saved,
        on=keys,
        how="outer",
        suffixes=("_expected", "_saved"),
        indicator=True,
        validate="one_to_one"
    )

    missing_rows = int(
        comparison["_merge"].eq("left_only").sum()
    )

    extra_rows = int(
        comparison["_merge"].eq("right_only").sum()
    )

    matched = comparison.loc[
        comparison["_merge"].eq("both")
    ].copy()

    results = [
        {
            "Table": name,
            "Field": "Missing rows",
            "Compared": len(expected),
            "Mismatches": missing_rows,
            "Null_Mismatches": 0
        },
        {
            "Table": name,
            "Field": "Unexpected rows",
            "Compared": len(saved),
            "Mismatches": extra_rows,
            "Null_Mismatches": 0
        }
    ]

    examples = {}

    non_key_columns = [
        c for c in expected.columns if c not in keys
    ]

    for col in non_key_columns:

        left = matched[f"{col}_expected"]
        right = matched[f"{col}_saved"]

        left_missing = left.isna().to_numpy()
        right_missing = right.isna().to_numpy()

        null_mismatch = left_missing != right_missing

        equal = (
            left_missing & right_missing
        )

        valid = ~left_missing & ~right_missing

        if (
            pd.api.types.is_numeric_dtype(left)
            and pd.api.types.is_numeric_dtype(right)
        ):
            a = left.to_numpy(dtype=float)
            b = right.to_numpy(dtype=float)

            equal[valid] = np.isclose(
                a[valid],
                b[valid],
                rtol=0,
                atol=1e-10,
                equal_nan=False
            )

        elif col in [
            "attack_start", "attack_end", "day"
        ]:
            a = normalize_date_355(left)
            b = normalize_date_355(right)

            equal[valid] = (
                a[valid].to_numpy()
                == b[valid].to_numpy()
            )

        else:
            equal[valid] = (
                left[valid].astype(str).to_numpy()
                == right[valid].astype(str).to_numpy()
            )

        mismatch = ~equal

        results.append({
            "Table": name,
            "Field": col,
            "Compared": len(matched),
            "Mismatches": int(mismatch.sum()),
            "Null_Mismatches": int(null_mismatch.sum())
        })

        if mismatch.any():

            positions = np.flatnonzero(
                mismatch
            )[:5]

            examples[col] = pd.DataFrame({
                **{
                    key: matched.iloc[positions][key].to_numpy()
                    for key in keys
                },
                "Expected": left.iloc[positions].to_numpy(),
                "Saved": right.iloc[positions].to_numpy()
            })

    return pd.DataFrame(results), examples, comparison


# ------------------------------------------------------------
# 12. Audit every saved table against reconstructed data.
# ------------------------------------------------------------

audit_results_355 = []
audit_examples_355 = {}

for name, expected, saved, keys in [
    (
        "Users",
        expected_users_355,
        saved_users_355,
        ["user"]
    ),
    (
        "Channel-Days",
        expected_channels_355,
        saved_channels_355,
        CHANNEL_KEYS_355
    ),
    (
        "User-Days",
        expected_days_355,
        saved_days_355,
        KEYS_355
    )
]:

    report, examples, comparison = audit_table_355(
        name, expected, saved, keys
    )

    audit_results_355.append(report)

    for field, example in examples.items():
        audit_examples_355[f"{name}: {field}"] = example

full_audit_355 = pd.concat(
    audit_results_355,
    ignore_index=True
)

print("\nD. VALUE-BY-VALUE GROUND TRUTH AUDIT")
display(full_audit_355)

# ------------------------------------------------------------
# 13. Summarize all three GT table comparisons.
# ------------------------------------------------------------

print("\nE. GROUND TRUTH AUDIT SUMMARY")

audit_summary_355 = (
    full_audit_355.groupby("Table", as_index=False)
    .agg(
        Checks=("Field", "size"),
        Mismatches=("Mismatches", "sum"),
        Null_Mismatches=("Null_Mismatches", "sum")
    )
)

display(audit_summary_355)

print("\nF. GROUND TRUTH LABEL DISTRIBUTION")

label_summary_355 = (
    expected_days_355
    .groupby("day_status")
    .agg(
        User_Days=("user", "size"),
        Injected_Events=("injected_events_total", "sum")
    )
    .reset_index()
)

display(label_summary_355)

# ------------------------------------------------------------
# 14. Additional logical and cross-table invariants.
# ------------------------------------------------------------

print("\nG. CROSS-TABLE MATHEMATICAL VALIDATION")

checks_355 = {}

checks_355["Users"] = (
    expected_users_355["user"].nunique() == 670
)

checks_355["Attack Days"] = (
    expected_days_355["is_attack_day"].sum() == 12636
)

checks_355["Off Days"] = (
    expected_days_355["day_status"].eq("Off").sum()
    == 2374
)

checks_355["Outside Schedule"] = (
    expected_days_355["day_status"]
    .eq("Outside Schedule").sum() == 54624
)

checks_355["Schedule Days"] = (
    expected_days_355["is_scheduled"].sum() == 15010
)

checks_355["Channel-Days"] = (
    len(expected_channels_355) == 14375
)

checks_355["Injected Events Agreement"] = (
    int(expected_users_355["injected_events_total"].sum())
    == int(expected_days_355["injected_events_total"].sum())
    == int(expected_channels_355["injected_count"].sum())
    == 42095
)

checks_355["Daily Channel Sums"] = (
    expected_days_355[
        ["injected_email", "injected_usb", "injected_http"]
    ].sum(axis=1).eq(
        expected_days_355["injected_events_total"]
    ).all()
)

checks_355["Planned vs Injected Channels"] = (
    expected_days_355["planned_channels"]
    .eq(expected_days_355["injected_channel_count"])
    .all()
)

checks_355["No Off-Plan Injection"] = (
    expected_days_355.loc[
        expected_days_355["day_status"].ne("Attack"),
        "injected_events_total"
    ].eq(0).all()
)

checks_355["Feature Label Separation"] = (
    {
        "scenario_id", "stealth_level", "day_status",
        "is_scheduled", "is_attack_day",
        "planned_channels", "injected_channel_count",
        "injected_email", "injected_usb", "injected_http",
        "injected_events_total", "injection_formula",
        "ground_truth_source"
    }.isdisjoint(saved_features_355.columns)
)

logical_report_355 = pd.DataFrame([
    {
        "Check": name,
        "Passed": bool(passed)
    }
    for name, passed in checks_355.items()
])

display(logical_report_355)

# ------------------------------------------------------------
# 15. Report discrepancy examples if necessary.
# ------------------------------------------------------------

print("\nH. GROUND TRUTH DISCREPANCY EXAMPLES")

if audit_examples_355:

    for name, examples in audit_examples_355.items():
        print(f"\n{name}")
        display(examples)

else:
    print("No saved-value discrepancies found.")

# ------------------------------------------------------------
# 16. Final independent audit decision.
# ------------------------------------------------------------

total_mismatches_355 = int(
    full_audit_355["Mismatches"].sum()
)

null_mismatches_355 = int(
    full_audit_355["Null_Mismatches"].sum()
)

logical_failures_355 = int(
    (~logical_report_355["Passed"]).sum()
)

expected_field_compares_355 = (
    670 * 11
    + 14375 * 8
    + 69634 * 11
)

actual_field_compares_355 = int(
    full_audit_355.loc[
        ~full_audit_355["Field"].isin(
            ["Missing rows", "Unexpected rows"]
        ),
        "Compared"
    ].sum()
)

print("\n" + "=" * 80)
print("STEP 3.5.5 — INDEPENDENT GROUND TRUTH AUDIT RESULTS")
print("=" * 80)

print("GT tables reconstructed: 3")
print("Expected non-key values:", expected_field_compares_355)
print("Non-key values compared:", actual_field_compares_355)
print("Saved-value mismatches:", total_mismatches_355)
print("Null-placement mismatches:", null_mismatches_355)
print("Logical invariant failures:", logical_failures_355)
print("Independent injected events: 42,095")
print("Files modified: NO")
print("Files exported: NO")

assert actual_field_compares_355 == expected_field_compares_355

assert total_mismatches_355 == 0, (
    "Ground Truth field audit FAILED."
)

assert null_mismatches_355 == 0, (
    "Ground Truth missing-value audit FAILED."
)

assert logical_failures_355 == 0, (
    "Ground Truth logical validation FAILED."
)

print("\nSTEP 3.5.5 RESULT: PASS")
print("User-Level Ground Truth: VERIFIED")
print("Channel-Day Ground Truth: VERIFIED")
print("Full User-Day Ground Truth: VERIFIED")
print("Scenario and stealth labels: VERIFIED")
print("Cross-table injection totals: VERIFIED")
print("Feature-label separation: VERIFIED")
print("Remaining Stage 3.5 checks: NOT YET PERFORMED")


STAGE 3.5 — STEP 3.5.5
INDEPENDENT GROUND TRUTH RECONSTRUCTION & AUDIT

A. SOURCE DISCOVERY AND LOADING: PASS

B. INDEPENDENT POPULATION
Users: 670
Eligible user-days: 69634

C. THREE GROUND TRUTH TABLES RECONSTRUCTED


,Dataset,Rows,Columns
0,User-Level,670,12
1,Channel-Day,14375,11
2,Full User-Day,69634,13



D. VALUE-BY-VALUE GROUND TRUTH AUDIT


,Table,Field,Compared,Mismatches,Null_Mismatches
0,Users,Missing rows,670,0,0
1,Users,Unexpected rows,670,0,0
2,Users,scenario_id,670,0,0
3,Users,stealth_level,670,0,0
4,Users,attack_start,670,0,0
5,Users,attack_end,670,0,0
6,Users,duration_days,670,0,0
7,Users,attack_days,670,0,0
8,Users,off_days,670,0,0
9,Users,planned_switches,670,0,0



E. GROUND TRUTH AUDIT SUMMARY


,Table,Checks,Mismatches,Null_Mismatches
0,Channel-Days,10,0,0
1,User-Days,13,0,0
2,Users,13,0,0



F. GROUND TRUTH LABEL DISTRIBUTION


,day_status,User_Days,Injected_Events
0,Attack,12636,42095
1,Off,2374,0
2,Outside Schedule,54624,0



G. CROSS-TABLE MATHEMATICAL VALIDATION


,Check,Passed
0,Users,True
1,Attack Days,True
2,Off Days,True
3,Outside Schedule,True
4,Schedule Days,True
5,Channel-Days,True
6,Injected Events Agreement,True
7,Daily Channel Sums,True
8,Planned vs Injected Channels,True
9,No Off-Plan Injection,True



H. GROUND TRUTH DISCREPANCY EXAMPLES
No saved-value discrepancies found.

STEP 3.5.5 — INDEPENDENT GROUND TRUTH AUDIT RESULTS
GT tables reconstructed: 3
Expected non-key values: 888344
Non-key values compared: 888344
Saved-value mismatches: 0
Null-placement mismatches: 0
Logical invariant failures: 0
Independent injected events: 42,095
Files modified: NO
Files exported: NO

STEP 3.5.5 RESULT: PASS
User-Level Ground Truth: VERIFIED
Channel-Day Ground Truth: VERIFIED
Full User-Day Ground Truth: VERIFIED
Scenario and stealth labels: VERIFIED
Cross-table injection totals: VERIFIED
Feature-label separation: VERIFIED
Remaining Stage 3.5 checks: NOT YET PERFORMED



### Step 3.5.6 — Statistical, Temporal & Adversarial Validation

**Objective:** Evaluate statistical consistency, historical baseline provenance, temporal causality, frozen context preservation, and the effectiveness of validation checks against intentional errors.

**Inputs:**
- Original CERT r6.2 User-Day records
- Original User Baseline
- Independently rebuilt behavioral features
- Saved Formula B feature and Ground Truth files

**Validation Groups:**

**A. Historical Baseline Provenance**
- Reconstruct pre-2011 eligible historical workdays.
- Recalculate active days, activity rates, positive-day means, and P50/P90/P95 for Email, USB, and HTTP.
- Compare the reconstructed historical statistics with the baseline file.

**B. Context Preservation**
- Check that historical payload-byte and other byte-related source columns remain unchanged after count-level injection.

**C. Stealth-Level Statistical Diagnostics**
- Analyze High, Medium, and Low separately.
- Measure positive injection coverage, P95 exceedance, and Final/P95 ratios.
- Check whether the median severity follows the intended High-to-Low progression.
- Report exceptions rather than hiding them.

**D. Temporal Causality**
- Select users from each stealth level.
- Recalculate all 30 new features using only historical prefixes ending at specific workdays.
- Compare the calculated values with the corresponding saved values.
- Verify that calculations at earlier dates do not require later observations.

**E. Intentional Fault Detection**
On temporary in-memory copies only:
- Corrupt a daily event count.
- Corrupt a behavioral feature.
- Replace a valid value with NaN.
- Flip an Attack label.
- Duplicate a User-Day key.

Verify that each corruption is detected.

**Validation Principle:** Statistical plausibility and mathematical correctness are separate. A correct Formula B implementation does not automatically guarantee realistic exfiltration or high ML detection performance.

**Safety:** Read-only. No files are modified or exported.

**Expected Outcome:** Detailed diagnostics and explicit PASS/FAIL results for each validation group.


In [8]:

# Stage 3 — Step 3.5.6:
# Statistical, Temporal, Edge-Case & Adversarial Validation
#
# Goal:
# 1. Recalculate historical baseline statistics.
# 2. Verify frozen byte-related context.
# 3. Examine stealth-level statistical behavior.
# 4. Independently recalculate features at past-only cutoffs.
# 5. Test intentional data corruption detection.
#
# STRICT READ-ONLY:
# No original or final files modified or exported.

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 78)
print("STAGE 3.5 — STEP 3.5.6")
print("STATISTICAL, TEMPORAL & ADVERSARIAL VALIDATION")
print("=" * 78)

# ------------------------------------------------------------
# 1. Confirm independently validated inputs exist.
# ------------------------------------------------------------

required_356 = [
    "PATHS_353",
    "PATHS_355",
    "rebuilt_353",
    "raw_353",
    "baseline_353",
    "expected_days_355",
    "expected_channels_355",
    "expected_users_355"
]

missing_356 = [
    name
    for name in required_356
    if name not in globals()
]

if missing_356:
    raise RuntimeError(
        "Run Steps 3.5.3–3.5.5 first. Missing: "
        + str(missing_356)
    )

# Load saved outputs directly from disk.
features_356 = pd.read_parquet(
    PATHS_353["features"]
)

days_356 = pd.read_parquet(
    PATHS_355["gt_days"]
)

channels_356 = pd.read_parquet(
    PATHS_355["gt_channels"]
)

users_356 = pd.read_parquet(
    PATHS_355["gt_users"]
)

# Independent source reconstructions.
source_356 = rebuilt_353.copy(deep=True)
natural_356 = raw_353.copy(deep=True)
baseline_356 = baseline_353.copy(deep=True)

KEYS_356 = ["user", "day"]
LEVELS_356 = ["High", "Medium", "Low"]

for frame in [
    features_356,
    days_356,
    channels_356,
    source_356,
    natural_356
]:
    frame["day"] = (
        pd.to_datetime(frame["day"])
        .astype("datetime64[ns]")
        .dt.normalize()
    )

for frame in [
    features_356,
    days_356,
    source_356,
    natural_356
]:
    frame.sort_values(
        KEYS_356,
        inplace=True
    )
    frame.reset_index(
        drop=True,
        inplace=True
    )

assert features_356.shape == (69634, 115)
assert days_356.shape == (69634, 13)

assert len(channels_356) == 14375
assert len(users_356) == 670

for frame in [
    days_356,
    source_356,
    natural_356
]:
    pd.testing.assert_frame_equal(
        features_356[KEYS_356],
        frame[KEYS_356],
        check_exact=True
    )

assert baseline_356["user"].is_unique
assert users_356["user"].is_unique

print("\nA. INPUT ALIGNMENT: PASS")


# ------------------------------------------------------------
# 2. Independently reconstruct pre-2011 historical data.
#
# Historical selection:
# - Before 2011-01-01
# - Non-weekend
# - At least one logon
# - Simulated user population
#
# This is a NEW provenance check: it tests baseline
# calculation, not just baseline-to-feature consistency.
# ------------------------------------------------------------

raw_history_356 = pd.read_parquet(
    PATHS_353["user_day"],
    columns=[
        "user",
        "day",
        "is_weekend",
        "logon_count",
        "external_email_count",
        "usb_copy_count",
        "www_upload_count"
    ]
)

raw_history_356["day"] = (
    pd.to_datetime(raw_history_356["day"])
    .astype("datetime64[ns]")
    .dt.normalize()
)

population_356 = set(
    users_356["user"]
)

history_356 = raw_history_356.loc[
    raw_history_356["user"].isin(population_356)
    & raw_history_356["day"].lt(
        pd.Timestamp("2011-01-01")
    )
    & raw_history_356["is_weekend"].eq(0)
    & raw_history_356["logon_count"].gt(0)
].copy()

assert len(history_356) > 0

assert not history_356.duplicated(
    KEYS_356
).any()

users_index_356 = pd.Index(
    sorted(population_356),
    name="user"
)

baseline_indexed_356 = (
    baseline_356
    .set_index("user")
    .reindex(users_index_356)
)

historical_workdays_356 = (
    history_356.groupby("user")
    .size()
    .reindex(
        users_index_356,
        fill_value=0
    )
)

baseline_audits_356 = []
baseline_examples_356 = {}

def check_historical_field_356(field, expected):

    if field not in baseline_indexed_356.columns:
        raise KeyError(
            f"Missing historical field: {field}"
        )

    actual = pd.to_numeric(
        baseline_indexed_356[field],
        errors="raise"
    ).to_numpy(dtype=float)

    recalculated = pd.to_numeric(
        expected.reindex(users_index_356),
        errors="raise"
    ).to_numpy(dtype=float)

    equal = np.isclose(
        recalculated,
        actual,
        rtol=0,
        atol=1e-8,
        equal_nan=True
    )

    mismatches = int((~equal).sum())

    baseline_audits_356.append({
        "Historical_Field": field,
        "Users_Checked": len(equal),
        "Mismatches": mismatches
    })

    if mismatches:
        positions = np.flatnonzero(
            ~equal
        )[:8]

        baseline_examples_356[field] = pd.DataFrame({
            "user": users_index_356[positions],
            "Recalculated": recalculated[positions],
            "Saved_Baseline": actual[positions]
        })


# Validate historical workday denominators.
check_historical_field_356(
    "baseline_workdays",
    historical_workdays_356
)

HISTORICAL_CHANNELS_356 = [
    (
        "Email",
        "external_email_count",
        "email_count",
        "email"
    ),
    (
        "USB",
        "usb_copy_count",
        "usb_copy",
        "usb"
    ),
    (
        "HTTP",
        "www_upload_count",
        "http_upload",
        "http_upload"
    )
]

for channel, count_col, prefix, active_prefix in (
    HISTORICAL_CHANNELS_356
):

    active_history = history_356.loc[
        history_356[count_col].gt(0),
        ["user", count_col]
    ]

    groups = active_history.groupby(
        "user"
    )[count_col]

    active_days = (
        groups.size()
        .reindex(users_index_356, fill_value=0)
    )

    active_rate = active_days.div(
        historical_workdays_356.replace(0, np.nan)
    )

    check_historical_field_356(
        f"{active_prefix}_active_days",
        active_days
    )

    check_historical_field_356(
        f"{active_prefix}_active_rate",
        active_rate
    )

    check_historical_field_356(
        f"{prefix}_mean_active",
        groups.mean()
    )

    for percentile, suffix in [
        (0.50, "p50"),
        (0.90, "p90"),
        (0.95, "p95")
    ]:

        independently_calculated = (
            groups.quantile(
                percentile,
                interpolation="linear"
            )
        )

        check_historical_field_356(
            f"{prefix}_{suffix}",
            independently_calculated
        )


baseline_report_356 = pd.DataFrame(
    baseline_audits_356
)

print("\nB. HISTORICAL BASELINE SOURCE RECONSTRUCTION")
display(baseline_report_356)

if baseline_examples_356:

    print("\nBASELINE DISCREPANCY EXAMPLES")

    print(
        "These require investigation. "
        "Check historical eligibility rules before "
        "treating them as calculation errors."
    )

    for field, examples in baseline_examples_356.items():

        print(f"\nField: {field}")
        display(examples)


# ------------------------------------------------------------
# 3. Check that byte-related context stayed unchanged.
#
# The approved injection changes event counts, not
# the original byte measurements.
# ------------------------------------------------------------

byte_columns_356 = [
    col
    for col in natural_356.columns
    if "byte" in col.lower()
    and col in features_356.columns
]

assert byte_columns_356, (
    "No byte-related columns found."
)

context_rows_356 = []

for column in byte_columns_356:

    original_values = pd.to_numeric(
        natural_356[column],
        errors="raise"
    ).to_numpy(dtype=float)

    final_values = pd.to_numeric(
        features_356[column],
        errors="raise"
    ).to_numpy(dtype=float)

    different = ~np.isclose(
        original_values,
        final_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    context_rows_356.append({
        "Frozen_Column": column,
        "Rows_Checked": len(original_values),
        "Differences": int(different.sum())
    })

context_audit_356 = pd.DataFrame(
    context_rows_356
)

print("\nC. FROZEN BYTE-CONTEXT VALIDATION")
display(context_audit_356)


# ------------------------------------------------------------
# 4. Evaluate statistical severity by stealth level.
#
# This is a descriptive simulation-quality diagnostic.
# It is NOT an ML detection-performance evaluation.
# ------------------------------------------------------------

channels_356["final_to_p95"] = (
    channels_356["final_count"]
    / channels_356["historical_p95"]
)

channels_356["natural_above_p95"] = (
    channels_356["natural_count"]
    > channels_356["historical_p95"]
)

channels_356["final_above_p95"] = (
    channels_356["final_count"]
    > channels_356["historical_p95"]
)

severity_rows_356 = []

for level in LEVELS_356:

    group = channels_356.loc[
        channels_356["stealth_level"].eq(level)
    ]

    severity_rows_356.append({
        "Stealth": level,
        "Channel_Days": len(group),
        "Positive_Injection_%": round(
            100 * group["injected_count"].gt(0).mean(),
            2
        ),
        "Natural_Above_P95_%": round(
            100 * group["natural_above_p95"].mean(),
            2
        ),
        "Final_Above_P95_%": round(
            100 * group["final_above_p95"].mean(),
            2
        ),
        "Median_Final_to_P95": round(
            group["final_to_p95"].median(),
            4
        ),
        "P90_Final_to_P95": round(
            group["final_to_p95"].quantile(0.90),
            4
        ),
        "Total_Injected": int(
            group["injected_count"].sum()
        )
    })

severity_report_356 = pd.DataFrame(
    severity_rows_356
)

print("\nD. STEALTH-LEVEL STATISTICAL DIAGNOSTICS")
display(severity_report_356)

medians_356 = severity_report_356[
    "Median_Final_to_P95"
].to_numpy(dtype=float)

stealth_order_356 = bool(
    np.all(np.diff(medians_356) > 0)
)

print(
    "High → Medium → Low median severity progression:",
    stealth_order_356
)

print(
    "P95 exceedances in High Stealth are reported "
    "as simulation-design exceptions, not removed."
)


# ------------------------------------------------------------
# 5. Causal prefix-only recomputation of all 30 features.
#
# Test users from High, Medium, and Low.
# For several cutoff dates, calculate features using
# only days available UP TO that cutoff.
#
# No future days, attack labels, or schedule information
# are used in this calculation.
# ------------------------------------------------------------

SPECS_356 = [
    (
        "email",
        "external_email_count",
        "email_count_p50",
        "email_count_p95",
        "email_active_rate",
        "email_count_mean_active"
    ),
    (
        "usb",
        "usb_copy_count",
        "usb_copy_p50",
        "usb_copy_p95",
        "usb_active_rate",
        "usb_copy_mean_active"
    ),
    (
        "http",
        "www_upload_count",
        "http_upload_p50",
        "http_upload_p95",
        "http_upload_active_rate",
        "http_upload_mean_active"
    )
]

WINDOWS_356 = [5, 10, 20, 30]


def direct_prefix_features_356(prefix, baseline_row):

    n = len(prefix)

    counts = np.column_stack([
        prefix[spec[1]].to_numpy(dtype=float)
        for spec in SPECS_356
    ])

    p50 = np.array([
        float(baseline_row[spec[2]])
        for spec in SPECS_356
    ])

    p95 = np.array([
        float(baseline_row[spec[3]])
        for spec in SPECS_356
    ])

    rates = np.array([
        float(baseline_row[spec[4]])
        for spec in SPECS_356
    ])

    means = np.array([
        float(baseline_row[spec[5]])
        for spec in SPECS_356
    ])

    valid_p95 = (
        np.isfinite(p95)
        & (p95 > 0)
    )

    normalized = np.divide(
        counts,
        p95,
        where=np.broadcast_to(valid_p95, (n, 3)),
        out=np.zeros((n, 3), dtype=float)
    )

    valid_p50 = (
        valid_p95
        & np.isfinite(p50)
        & (p50 > 0)
        & (p95 >= p50)
    )

    elevated = (
        (counts > p50)
        & valid_p50
    )

    positive_deviation = np.maximum(
        0,
        counts - p50
    )

    daily_deviation = np.divide(
        positive_deviation,
        np.maximum(1, p95 - p50),
        where=np.broadcast_to(valid_p50, (n, 3)),
        out=np.zeros((n, 3), dtype=float)
    ).sum(axis=1)

    daily_intensity = (
        normalized.sum(axis=1)
        / valid_p95.sum()
    )

    historical_weight = np.divide(
        np.nan_to_num(rates)
        * np.nan_to_num(means),
        p95,
        where=valid_p95,
        out=np.zeros(3, dtype=float)
    )

    assert historical_weight.sum() > 0

    historical_share = (
        historical_weight
        / historical_weight.sum()
    )

    # Observed daily dominant channel.
    # -1 indicates an inactive or tied day.
    dominant = []

    for values in normalized:

        maximum = values.max()

        tied = np.isclose(
            values,
            maximum,
            rtol=1e-9,
            atol=1e-12
        )

        if maximum <= 0 or tied.sum() != 1:
            dominant.append(-1)
        else:
            dominant.append(int(np.argmax(values)))

    dominant = np.asarray(dominant)

    results = {}

    for window in WINDOWS_356:

        prefixes = [
            "cumulative_deviation",
            "combined_channel_intensity",
            "dominant_channel_switches",
            "channel_distribution_shift",
            "consecutive_elevated_days",
            "cross_channel_deviation_days"
        ]

        names = [
            f"{name}_{window}wd"
            for name in prefixes
        ]

        # Incomplete windows must remain missing.
        if n < window:

            for name in names:
                results[name] = np.nan

            if window in [20, 30]:

                for name in [
                    "external_email_count",
                    "usb_copy_count",
                    "http_upload_count"
                ]:
                    results[f"{name}_{window}wd"] = np.nan

            continue

        segment = slice(
            n - window,
            n
        )

        # Cumulative deviation.
        results[names[0]] = float(
            daily_deviation[segment].sum()
        )

        # Combined intensity.
        results[names[1]] = float(
            daily_intensity[segment].mean()
        )

        # Dominant-channel switching.
        observed = dominant[segment]
        observed = observed[observed >= 0]

        if len(observed) > 1:
            switches = np.count_nonzero(
                observed[1:] != observed[:-1]
            )
        else:
            switches = 0

        results[names[2]] = float(switches)

        # Channel distribution shift (TVD).
        channel_totals = normalized[
            segment
        ].sum(axis=0)

        total = float(
            channel_totals.sum()
        )

        if total > 0:

            results[names[3]] = float(
                0.5 * np.abs(
                    channel_totals / total
                    - historical_share
                ).sum()
            )

        else:
            results[names[3]] = np.nan

        # Longest consecutive elevated-day sequence.
        flags = elevated[
            segment
        ].any(axis=1)

        longest = 0
        run = 0

        for flag in flags:

            if flag:
                run += 1
            else:
                run = 0

            longest = max(
                longest,
                run
            )

        results[names[4]] = float(longest)

        # Cross-channel deviation days.
        results[names[5]] = float(
            (
                elevated[segment].sum(axis=1)
                >= 2
            ).sum()
        )

        # Additional 20/30 WD channel totals.
        if window in [20, 30]:

            for index, name in enumerate([
                "external_email_count",
                "usb_copy_count",
                "http_upload_count"
            ]):

                results[f"{name}_{window}wd"] = float(
                    counts[segment, index].sum()
                )

    assert len(results) == 30

    return results


# Select three users from each stealth category.
sampled_users_356 = []

for level in LEVELS_356:

    candidates = sorted(
        users_356.loc[
            users_356["stealth_level"].eq(level),
            "user"
        ].astype(str)
    )

    sampled_users_356.extend(
        candidates[:3]
    )

assert len(sampled_users_356) == 9

saved_lookup_356 = (
    features_356
    .set_index(KEYS_356)
)

baseline_lookup_356 = (
    baseline_356
    .set_index("user")
)

prefix_rows_356 = []

for user in sampled_users_356:

    user_frame = source_356.loc[
        source_356["user"].eq(user)
    ].sort_values("day").reset_index(drop=True)

    length = len(user_frame)

    cutoffs = sorted(set([
        5,
        10,
        20,
        30,
        min(length, 45),
        length
    ]))

    for cutoff in cutoffs:

        if cutoff > length:
            continue

        prefix = user_frame.iloc[
            :cutoff
        ]

        independently_calculated = (
            direct_prefix_features_356(
                prefix,
                baseline_lookup_356.loc[user]
            )
        )

        endpoint = prefix[
            "day"
        ].iloc[-1]

        saved_row = saved_lookup_356.loc[
            (user, endpoint)
        ]

        for feature, expected in (
            independently_calculated.items()
        ):

            actual = float(
                saved_row[feature]
            )

            passed = bool(
                np.isclose(
                    expected,
                    actual,
                    rtol=1e-10,
                    atol=1e-8,
                    equal_nan=True
                )
            )

            prefix_rows_356.append({
                "user": user,
                "Prefix_Workdays": cutoff,
                "Feature": feature,
                "Passed": passed,
                "Expected": expected,
                "Saved": actual
            })


prefix_report_356 = pd.DataFrame(
    prefix_rows_356
)

print("\nE. PAST-ONLY FEATURE RECOMPUTATION")

print("Users tested:", len(sampled_users_356))
print("Feature comparisons:", len(prefix_report_356))

print(
    "Mismatches:",
    int((~prefix_report_356["Passed"]).sum())
)

if not prefix_report_356["Passed"].all():

    display(
        prefix_report_356.loc[
            ~prefix_report_356["Passed"]
        ].head(15)
    )


# ------------------------------------------------------------
# 6. Intentionally corrupt IN-MEMORY COPIES.
#
# These tests verify that validation comparisons
# can detect errors, rather than only confirming
# already correct datasets.
#
# Never write the altered copies to disk.
# ------------------------------------------------------------

def count_numeric_mismatches_356(left, right):

    equal = np.isclose(
        left.to_numpy(dtype=float),
        right.to_numpy(dtype=float),
        rtol=0,
        atol=1e-10,
        equal_nan=True
    )

    return int((~equal).sum())


fault_rows_356 = []

# Test 1: Corrupt a daily event count.
corrupted = features_356.copy(deep=True)

corrupted.loc[
    0, "external_email_count"
] += 1

detected = (
    count_numeric_mismatches_356(
        corrupted["external_email_count"],
        features_356["external_email_count"]
    ) == 1
)

fault_rows_356.append({
    "Mutation": "Change one daily event count",
    "Detected": detected
})


# Test 2: Corrupt a calculated behavioral feature.
corrupted = features_356.copy(deep=True)

feature = "cumulative_deviation_30wd"

position = int(
    np.flatnonzero(
        corrupted[feature].notna().to_numpy()
    )[0]
)

corrupted.loc[
    position, feature
] += 1.0

detected = (
    count_numeric_mismatches_356(
        corrupted[feature],
        features_356[feature]
    ) == 1
)

fault_rows_356.append({
    "Mutation": "Change one behavioral feature",
    "Detected": detected
})


# Test 3: Insert an unexpected NaN.
corrupted = features_356.copy(deep=True)

corrupted.loc[
    position, feature
] = np.nan

detected = (
    count_numeric_mismatches_356(
        corrupted[feature],
        features_356[feature]
    ) == 1
)

fault_rows_356.append({
    "Mutation": "Replace a valid value with NaN",
    "Detected": detected
})


# Test 4: Flip one Ground Truth attack label.
corrupted_labels = days_356.copy(deep=True)

attack_position = int(
    np.flatnonzero(
        corrupted_labels[
            "is_attack_day"
        ].eq(1).to_numpy()
    )[0]
)

corrupted_labels.loc[
    attack_position, "is_attack_day"
] = 0

detected = (
    int(
        (
            corrupted_labels["is_attack_day"]
            != days_356["is_attack_day"]
        ).sum()
    ) == 1
)

fault_rows_356.append({
    "Mutation": "Flip one Attack label",
    "Detected": detected
})


# Test 5: Duplicate a primary key.
corrupted_keys = pd.concat(
    [
        days_356,
        days_356.iloc[[0]]
    ],
    ignore_index=True
)

detected = bool(
    corrupted_keys.duplicated(
        KEYS_356
    ).any()
)

fault_rows_356.append({
    "Mutation": "Duplicate a User-Day key",
    "Detected": detected
})


fault_report_356 = pd.DataFrame(
    fault_rows_356
)

print("\nF. INTENTIONAL FAULT DETECTION")
display(fault_report_356)

print(
    "All corruptions were performed on "
    "temporary in-memory copies only."
)


# ------------------------------------------------------------
# 7. Final independent QA decision.
# ------------------------------------------------------------

checks_356 = {
    "Historical baseline matches pre-2011 source": bool(
        baseline_report_356["Mismatches"].eq(0).all()
    ),

    "Frozen byte columns remain unchanged": bool(
        context_audit_356["Differences"].eq(0).all()
    ),

    "Stealth medians progress High to Low": (
        stealth_order_356
    ),

    "All scheduled channel injections are positive": bool(
        channels_356["injected_count"].gt(0).all()
    ),

    "Past-only feature calculations match saved results": bool(
        prefix_report_356["Passed"].all()
    ),

    "Every intentional corruption is detected": bool(
        fault_report_356["Detected"].all()
    ),

    "Total injected events remain 42,095": (
        int(channels_356["injected_count"].sum())
        == 42095
    )
}

final_checks_356 = pd.DataFrame([
    {
        "Check": name,
        "Passed": passed
    }
    for name, passed in checks_356.items()
])

print("\nG. FINAL STATISTICAL AND TEMPORAL QA")
display(final_checks_356)

print("\n" + "=" * 78)
print("STEP 3.5.6 — FINAL AUDIT RESULTS")
print("=" * 78)

print(
    "Historical fields checked:",
    len(baseline_report_356)
)

print(
    "Historical baseline mismatches:",
    int(baseline_report_356["Mismatches"].sum())
)

print(
    "Frozen context mismatches:",
    int(context_audit_356["Differences"].sum())
)

print(
    "Past-only feature comparison mismatches:",
    int((~prefix_report_356["Passed"]).sum())
)

print(
    "Intentional faults detected:",
    int(fault_report_356["Detected"].sum()),
    "/",
    len(fault_report_356)
)

print("Files modified: NO")
print("Files exported: NO")

failed_356 = final_checks_356.loc[
    ~final_checks_356["Passed"],
    "Check"
].tolist()

if failed_356:

    print("\nSTEP 3.5.6 RESULT: INVESTIGATE")
    print("Failed checks:", failed_356)

    raise AssertionError(
        "Investigate discrepancies before proceeding."
    )

print("\nSTEP 3.5.6 RESULT: PASS")
print("Historical baseline provenance: VERIFIED")
print("Frozen byte-related context: VERIFIED")
print("Statistical stealth progression: VERIFIED")
print("Past-only feature calculations: VERIFIED")
print("Intentional error detection: VERIFIED")
print("Remaining Stage 3.5 step: FINAL QA REPORT")


STAGE 3.5 — STEP 3.5.6
STATISTICAL, TEMPORAL & ADVERSARIAL VALIDATION

A. INPUT ALIGNMENT: PASS

B. HISTORICAL BASELINE SOURCE RECONSTRUCTION


,Historical_Field,Users_Checked,Mismatches
0,baseline_workdays,670,0
1,email_active_days,670,0
2,email_active_rate,670,0
3,email_count_mean_active,670,0
4,email_count_p50,670,0
5,email_count_p90,670,0
6,email_count_p95,670,0
7,usb_active_days,670,0
8,usb_active_rate,670,0
9,usb_copy_mean_active,670,0



C. FROZEN BYTE-CONTEXT VALIDATION


,Frozen_Column,Rows_Checked,Differences
0,total_sent_email_bytes,69634,0
1,external_email_bytes,69634,0



D. STEALTH-LEVEL STATISTICAL DIAGNOSTICS


,Stealth,Channel_Days,Positive_Injection_%,Natural_Above_P95_%,Final_Above_P95_%,Median_Final_to_P95,P90_Final_to_P95,Total_Injected
0,High,4413,100.0,3.56,25.90,0.75,1.5,7236
1,Medium,5023,100.0,3.80,39.50,1.00,2.0,13759
2,Low,4939,100.0,3.97,59.89,1.25,2.5,21100


High → Medium → Low median severity progression: True
P95 exceedances in High Stealth are reported as simulation-design exceptions, not removed.

E. PAST-ONLY FEATURE RECOMPUTATION
Users tested: 9
Feature comparisons: 1620
Mismatches: 0

F. INTENTIONAL FAULT DETECTION


,Mutation,Detected
0,Change one daily event count,True
1,Change one behavioral feature,True
2,Replace a valid value with NaN,True
3,Flip one Attack label,True
4,Duplicate a User-Day key,True


All corruptions were performed on temporary in-memory copies only.

G. FINAL STATISTICAL AND TEMPORAL QA


,Check,Passed
0,Historical baseline matches pre-2011 source,True
1,Frozen byte columns remain unchanged,True
2,Stealth medians progress High to Low,True
3,All scheduled channel injections are positive,True
4,Past-only feature calculations match saved res...,True
5,Every intentional corruption is detected,True
6,"Total injected events remain 42,095",True



STEP 3.5.6 — FINAL AUDIT RESULTS
Historical fields checked: 19
Historical baseline mismatches: 0
Frozen context mismatches: 0
Past-only feature comparison mismatches: 0
Intentional faults detected: 5 / 5
Files modified: NO
Files exported: NO

STEP 3.5.6 RESULT: PASS
Historical baseline provenance: VERIFIED
Frozen byte-related context: VERIFIED
Statistical stealth progression: VERIFIED
Past-only feature calculations: VERIFIED
Intentional error detection: VERIFIED
Remaining Stage 3.5 step: FINAL QA REPORT



### Step 3.5.7 — Final Independent QA Report

**Objective:** Consolidate and independently verify the results of all six completed Stage 3.5 audit steps, then generate a reproducible final quality-assurance report.

**Audit Coverage:**

1. Independent Formula B injection recalculation.
2. Scenario catalog and attack scheduling validation.
3. Independent reconstruction of the original 85 columns.
4. Independent reconstruction of all 30 new features.
5. Independent reconstruction of three Ground Truth tables.
6. Historical baseline, statistical, temporal, and fault-detection validation.

**Final Quality Gates:**
- All mandatory checks must pass.
- Zero unexplained numerical or label mismatches.
- Zero mathematical identity violations.
- Zero scheduling violations.
- Zero missing-value placement mismatches.
- Successful temporal-prefix comparisons.
- Successful intentional fault-detection tests.
- Final source and output fingerprints must remain unchanged since the initial audit.

**Generated Report:**
- Executive summary and validation scope.
- Stage-by-stage evidence tables.
- Numerical comparison coverage.
- Stealth-level diagnostics and charts.
- Rolling-window coverage.
- Historical baseline provenance.
- Intentional fault-detection results.
- Source-file fingerprint manifest.
- Limitations and remaining ML validation responsibilities.

**Outputs:**
- Self-contained HTML QA report.
- Detailed validation-check CSV.
- SHA-256 file manifest CSV.
- Machine-readable QA summary JSON.

**Safety:**
- Do not modify original source datasets.
- Do not modify exported features or Ground Truth.
- Save reports only to a newly created report directory.
- Never overwrite existing reports.
- Do not claim an overall PASS if any mandatory gate fails.

**Expected Outcome:** A reproducible Stage 3.5 final QA report documenting the independent validation of the MUBSIR attack simulation.


In [12]:

# Stage 3 — Step 3.5.7: Final Independent QA Report
#
# Goal:
# Consolidate all six independent validation stages,
# verify their results and file fingerprints, and
# generate a comprehensive QA report.
#
# SAFETY:
# - No source data modifications.
# - No Ground Truth or feature modifications.
# - Never overwrite previous QA reports.
# - Report generation only after mandatory checks pass.

from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
from io import BytesIO
from html import escape
import hashlib
import base64
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

print("=" * 80)
print("STAGE 3.5 — STEP 3.5.7")
print("FINAL INDEPENDENT QUALITY ASSURANCE REPORT")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify all prior audit results are available.
# ------------------------------------------------------------

REQUIRED_357 = [
    "ALL_PATHS_351",
    "fingerprints_351",
    "channel_audit_351",
    "daily_audit_351",
    "catalog_summary_352",
    "user_audit_352",
    "day_totals_352",
    "identity_audit_353",
    "feature_audit_353",
    "feature_audit_354",
    "coverage_354",
    "full_audit_355",
    "logical_report_355",
    "baseline_report_356",
    "context_audit_356",
    "severity_report_356",
    "prefix_report_356",
    "fault_report_356",
    "final_checks_356",
    "PATHS_353"
]

missing_357 = [
    name for name in REQUIRED_357
    if name not in globals()
]

if missing_357:
    raise RuntimeError(
        "Complete Steps 3.5.1–3.5.6 first. "
        f"Missing objects: {missing_357}"
    )

ROOT_357 = Path(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration"
)

REPORT_ROOT_357 = (
    ROOT_357
    / "Attack_Simulation"
    / "validation_reports"
)

# ------------------------------------------------------------
# 2. Independently collect quantitative audit evidence.
# ------------------------------------------------------------

original_comparisons_357 = int(
    feature_audit_353["Compared_Rows"].sum()
)

new_comparisons_357 = int(
    feature_audit_354["Compared_Rows"].sum()
)

gt_comparisons_357 = int(
    full_audit_355.loc[
        ~full_audit_355["Field"].isin(
            ["Missing rows", "Unexpected rows"]
        ),
        "Compared"
    ].sum()
)

total_comparisons_357 = (
    original_comparisons_357
    + new_comparisons_357
    + gt_comparisons_357
)

historical_comparisons_357 = int(
    baseline_report_356["Users_Checked"].sum()
)

temporal_comparisons_357 = len(prefix_report_356)

faults_tested_357 = len(fault_report_356)

faults_detected_357 = int(
    fault_report_356["Detected"].sum()
)

assert original_comparisons_357 == 5779622
assert new_comparisons_357 == 2089020
assert gt_comparisons_357 == 888344
assert total_comparisons_357 == 8756986

assert historical_comparisons_357 == 12730
assert temporal_comparisons_357 == 1620
assert faults_tested_357 == 5

print("\nA. INDEPENDENT VALIDATION COVERAGE")

coverage_summary_357 = pd.DataFrame([
    {
        "Validation": "Original 83 non-key columns",
        "Comparisons": original_comparisons_357,
        "Mismatches": int(
            feature_audit_353["Mismatches"].sum()
        )
    },
    {
        "Validation": "30 new behavioral features",
        "Comparisons": new_comparisons_357,
        "Mismatches": int(
            feature_audit_354["Value_Mismatches"].sum()
        )
    },
    {
        "Validation": "Three Ground Truth tables",
        "Comparisons": gt_comparisons_357,
        "Mismatches": int(
            full_audit_355["Mismatches"].sum()
        )
    }
])

display(coverage_summary_357)

# ------------------------------------------------------------
# 3. Re-evaluate EVERY mandatory quality gate.
#
# These statuses are derived from previous audit
# results, not entered manually as PASS strings.
# ------------------------------------------------------------

GATES_357 = []

def add_gate_357(stage, check, passed, evidence):
    GATES_357.append({
        "Stage": stage,
        "Check": check,
        "Passed": bool(passed),
        "Evidence": str(evidence)
    })


# Stage 3.5.1
add_gate_357(
    "3.5.1",
    "Channel-Day injection reconciliation",
    channel_audit_351["Mismatches"].eq(0).all(),
    "All saved channel-day quantities match independent calculations"
)

add_gate_357(
    "3.5.1",
    "Daily channel-count reconciliation",
    daily_audit_351["Mismatched_Rows"].eq(0).all(),
    "69,634 user-days compared for Email, USB and HTTP"
)

# Stage 3.5.2
add_gate_357(
    "3.5.2",
    "Scenario catalog mathematics",
    catalog_summary_352["Failures"].eq(0).all(),
    "1,143 approved scenarios checked"
)

add_gate_357(
    "3.5.2",
    "Scheduling and channel validity",
    (
        len(user_audit_352) == 670
        and user_audit_352["status"].eq("PASS").all()
    ),
    "670 independently checked user schedules"
)

add_gate_357(
    "3.5.2",
    "Day-type totals",
    day_totals_352["Difference"].eq(0).all(),
    "Off, Single, Two and Three-Channel totals"
)

# Stage 3.5.3
add_gate_357(
    "3.5.3",
    "Original feature values",
    feature_audit_353["Mismatches"].eq(0).all(),
    f"{original_comparisons_357:,} value comparisons"
)

add_gate_357(
    "3.5.3",
    "Original feature null placement",
    feature_audit_353["Null_Mismatches"].eq(0).all(),
    "All 83 non-key columns"
)

add_gate_357(
    "3.5.3",
    "Aggregate event-count identities",
    identity_audit_353["Violations"].eq(0).all(),
    "Email, File and HTTP event identities"
)

# Stage 3.5.4
add_gate_357(
    "3.5.4",
    "Thirty new feature values",
    feature_audit_354["Value_Mismatches"].eq(0).all(),
    f"{new_comparisons_357:,} value comparisons"
)

add_gate_357(
    "3.5.4",
    "New feature null placement",
    feature_audit_354["Null_Mismatches"].eq(0).all(),
    "5, 10, 20 and 30 WD completeness"
)

# Stage 3.5.5
add_gate_357(
    "3.5.5",
    "Ground Truth row and field values",
    full_audit_355["Mismatches"].eq(0).all(),
    f"{gt_comparisons_357:,} non-key comparisons"
)

add_gate_357(
    "3.5.5",
    "Ground Truth null placement",
    full_audit_355["Null_Mismatches"].eq(0).all(),
    "All three Ground Truth tables"
)

add_gate_357(
    "3.5.5",
    "Ground Truth logical invariants",
    logical_report_355["Passed"].all(),
    "All cross-table relationships and label checks"
)

# Stage 3.5.6
add_gate_357(
    "3.5.6",
    "Historical baseline provenance",
    baseline_report_356["Mismatches"].eq(0).all(),
    f"{historical_comparisons_357:,} historical comparisons"
)

add_gate_357(
    "3.5.6",
    "Frozen byte-related context",
    context_audit_356["Differences"].eq(0).all(),
    "Original email-byte measurements unchanged"
)

add_gate_357(
    "3.5.6",
    "Past-only feature recalculation",
    prefix_report_356["Passed"].all(),
    f"{temporal_comparisons_357:,} prefix comparisons"
)

add_gate_357(
    "3.5.6",
    "Intentional corruption detection",
    fault_report_356["Detected"].all(),
    f"{faults_detected_357}/{faults_tested_357} faults detected"
)

add_gate_357(
    "3.5.6",
    "Final statistical QA checks",
    final_checks_356["Passed"].all(),
    "Historical, temporal and stealth diagnostics"
)

checks_357 = pd.DataFrame(GATES_357)

print("\nB. FINAL MANDATORY QUALITY GATES")
display(checks_357)

# ------------------------------------------------------------
# 4. Recalculate file SHA-256 hashes and compare
#    them with hashes captured at the start of Stage 3.5.
# ------------------------------------------------------------

def sha256_report_357(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(8 * 1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


manifest_paths_357 = dict(ALL_PATHS_351)

baseline_parquet_357 = (
    PATHS_353["baseline_parquet"]
)

if baseline_parquet_357.is_file():
    manifest_paths_357[
        "Historical Baseline Parquet"
    ] = baseline_parquet_357

initial_hashes_357 = (
    fingerprints_351.set_index("Dataset")["SHA256"]
    .to_dict()
)

manifest_rows_357 = []

for name, path in manifest_paths_357.items():

    assert path.is_file(), (
        f"File missing during final audit: {path}"
    )

    final_hash = sha256_report_357(path)

    initial_hash = initial_hashes_357.get(name)

    unchanged = (
        final_hash == initial_hash
        if initial_hash is not None
        else None
    )

    manifest_rows_357.append({
        "Dataset": name,
        "File": str(path),
        "Size_Bytes": int(path.stat().st_size),
        "Initial_SHA256": initial_hash,
        "Final_SHA256": final_hash,
        "Unchanged_Since_3_5_1": unchanged
    })

manifest_357 = pd.DataFrame(
    manifest_rows_357
)

known_hashes_357 = manifest_357.loc[
    manifest_357["Initial_SHA256"].notna()
]

hashes_unchanged_357 = bool(
    known_hashes_357[
        "Unchanged_Since_3_5_1"
    ].eq(True).all()
)

add_gate_357(
    "3.5.7",
    "Input/output file integrity",
    hashes_unchanged_357,
    "SHA-256 matches the initial Stage 3.5 fingerprints"
)

checks_357 = pd.DataFrame(GATES_357)

print("\nC. FILE INTEGRITY MANIFEST")

display(
    manifest_357[
        [
            "Dataset",
            "Size_Bytes",
            "Unchanged_Since_3_5_1"
        ]
    ]
)

# ------------------------------------------------------------
# 5. Validate overall status BEFORE saving the report.
# ------------------------------------------------------------

failed_checks_357 = checks_357.loc[
    ~checks_357["Passed"]
]

print("\nD. FINAL AUDIT DECISION")

print("Mandatory checks:", len(checks_357))
print("Passed:", int(checks_357["Passed"].sum()))
print("Failed:", len(failed_checks_357))

if not failed_checks_357.empty:

    display(failed_checks_357)

    raise AssertionError(
        "Final QA report export blocked because "
        "one or more mandatory checks failed."
    )

print("All mandatory quality gates: PASS")

# ------------------------------------------------------------
# 6. Prepare evidence tables and descriptive statistics.
# ------------------------------------------------------------

stage_summary_357 = (
    checks_357.groupby("Stage", as_index=False)
    .agg(
        Checks=("Check", "size"),
        Passed=("Passed", "sum")
    )
)

stage_summary_357["Status"] = np.where(
    stage_summary_357["Checks"]
    == stage_summary_357["Passed"],
    "PASS",
    "FAIL"
)

severity_357 = severity_report_356.copy()
windows_357 = coverage_354.copy()

windows_357["Complete_%"] = (
    100
    * windows_357["Complete_Windows"]
    / (
        windows_357["Complete_Windows"]
        + windows_357["Incomplete_Windows"]
    )
).round(2)

print("\nE. SUMMARY BY VALIDATION STEP")
display(stage_summary_357)

print("\nF. STEALTH DIAGNOSTICS")
display(severity_357)

print("\nG. ROLLING WINDOW COVERAGE")
display(windows_357)

# ------------------------------------------------------------
# 7. Create self-contained, embedded report charts.
#
# Figures are generated in memory, then embedded
# directly into a single HTML report.
# ------------------------------------------------------------

def embed_figure_357(fig):

    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=160,
        bbox_inches="tight"
    )

    plt.close(fig)

    return (
        "data:image/png;base64,"
        + base64.b64encode(
            buffer.getvalue()
        ).decode("ascii")
    )


# Chart 1: P95 exceedance by stealth.
fig1, ax1 = plt.subplots(figsize=(7, 3.5))

ax1.bar(
    severity_357["Stealth"],
    severity_357["Final_Above_P95_%"]
)

ax1.set_ylabel("Channel-days above P95 (%)")
ax1.set_ylim(0, 100)
ax1.set_title("Final P95 Exceedance by Stealth Level")

for i, value in enumerate(
    severity_357["Final_Above_P95_%"]
):
    ax1.text(
        i,
        value + 2,
        f"{value:.2f}%",
        ha="center"
    )

chart_p95_357 = embed_figure_357(fig1)


# Chart 2: Median final activity / historical P95.
fig2, ax2 = plt.subplots(figsize=(7, 3.5))

ax2.bar(
    severity_357["Stealth"],
    severity_357["Median_Final_to_P95"]
)

ax2.axhline(
    1.0,
    linestyle="--",
    linewidth=1,
    label="Historical P95"
)

ax2.set_ylabel("Median Final Count / P95")
ax2.set_title("Median Injection Severity by Stealth")
ax2.legend()

chart_ratio_357 = embed_figure_357(fig2)


# Chart 3: Complete rolling-window coverage.
fig3, ax3 = plt.subplots(figsize=(7, 3.5))

ax3.bar(
    windows_357["Window_WD"].astype(str),
    windows_357["Complete_%"]
)

ax3.set_ylim(0, 100)
ax3.set_ylabel("Complete windows (%)")
ax3.set_xlabel("Rolling window (workdays)")
ax3.set_title("Feature Window Completeness")

for i, value in enumerate(
    windows_357["Complete_%"]
):
    ax3.text(
        i,
        value + 1,
        f"{value:.2f}%",
        ha="center"
    )

chart_windows_357 = embed_figure_357(fig3)


# ------------------------------------------------------------
# 8. Build the detailed, self-contained HTML QA report.
# ------------------------------------------------------------

def html_table_357(frame):

    return frame.to_html(
        index=False,
        border=0,
        classes="audit-table",
        escape=True
    )


generated_utc_357 = (
    datetime.now(timezone.utc)
    .strftime("%Y-%m-%d %H:%M:%S UTC")
)

LIMITATIONS_357 = [
    (
        "The simulated injections modify aggregated event "
        "counts. Individual synthetic event IDs, event "
        "timestamps, and payload transfers were not generated."
    ),
    (
        "The 42,095 injected events represent additional "
        "activity counts, not verified exfiltrated bytes."
    ),
    (
        "The historical baseline was verified against "
        "pre-2011 aggregated User-Day data; raw event-level "
        "reconstruction remains outside this audit scope."
    ),
    (
        "Temporal prefix tests independently checked "
        "9 users and 1,620 feature comparisons; this does "
        "not establish the absence of leakage in future "
        "machine-learning train/test splitting."
    ),
    (
        "The five mutation tests demonstrate detection "
        "of the five tested errors only, not all possible "
        "data-corruption or semantic errors."
    ),
    (
        "High-stealth channel-days may exceed historical "
        "P95. These exceedances are documented and are "
        "not silently removed."
    ),
    (
        "An Off or Outside Schedule label means no "
        "synthetic injection was made on that day. "
        "It does not independently prove benign intent."
    ),
    (
        "Feature correctness and statistical progression "
        "do not establish machine-learning detection "
        "performance or real-world attack realism."
    )
]

limitations_html_357 = "".join(
    f"<li>{escape(item)}</li>"
    for item in LIMITATIONS_357
)

html_357 = f"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>MUBSIR — Stage 3.5 Independent QA Report</title>
<style>
body {{
    font-family: "Segoe UI", Arial, sans-serif;
    max-width: 1100px;
    margin: 38px auto;
    padding: 0 24px;
    line-height: 1.6;
    color: #202936;
    background: #ffffff;
}}
h1, h2, h3 {{ color: #153d57; }}
h1 {{ border-bottom: 3px solid #153d57; padding-bottom: 14px; }}
h2 {{ margin-top: 38px; border-bottom: 1px solid #dddddd; }}
.meta {{ color: #657383; font-size: 14px; }}
.metric {{
    display: inline-block;
    padding: 14px 24px;
    margin: 8px;
    border: 1px solid #d6e0e7;
    border-radius: 8px;
    min-width: 180px;
}}
.metric strong {{ font-size: 23px; display: block; }}
.audit-table {{
    width: 100%;
    border-collapse: collapse;
    font-size: 13px;
    margin: 18px 0 28px;
}}
.audit-table th, .audit-table td {{
    border: 1px solid #dce3e9;
    padding: 8px 10px;
    text-align: left;
    word-break: break-word;
}}
.audit-table th {{ background: #eaf0f5; }}
.audit-table tr:nth-child(even) {{ background: #f8fafc; }}
figure {{ margin: 16px 0 28px; }}
figure img {{ max-width: 100%; height: auto; }}
.note {{ color: #495869; }}
footer {{
    margin-top: 48px;
    padding-top: 12px;
    border-top: 1px solid #ccc;
    color: #657383;
    font-size: 12px;
}}
</style>
</head>
<body>

<h1>MUBSIR — Independent Attack Simulation QA</h1>

<p class="meta">
Stage 3.5 Final Report |
Generated: {escape(generated_utc_357)} |
Selected Formula: B_Natural_Addition
</p>

<h2>1. Executive Summary</h2>

<p>
The Stage 3.5 independent audit verified the
mathematical construction and saved consistency of
the CERT r6.2 synthetic attack simulation.
All mandatory checks passed within their defined scope.
</p>

<div class="metric">
<strong>{total_comparisons_357:,}</strong>
Feature and GT field comparisons
</div>

<div class="metric">
<strong>0</strong>
Unexplained mismatches
</div>

<div class="metric">
<strong>42,095</strong>
Injected event counts
</div>

<div class="metric">
<strong>670</strong>
Simulated users
</div>

<h2>2. Scope and Audit Method</h2>

<p>
The validation separately reconstructed Formula B,
attack scheduling, original and newly engineered
behavioral features, and all three Ground Truth datasets.
Original source datasets and exported outputs were loaded
from disk. Results were compared at user, user-day,
and user-day-channel levels.
</p>

<h3>Audit Stage Results</h3>
{html_table_357(stage_summary_357)}

<h3>Mandatory Quality Gates</h3>
{html_table_357(checks_357)}

<h2>3. Numerical Reconciliation</h2>

{html_table_357(coverage_summary_357)}

<p>
The values above exclude user/day primary-key comparisons,
which were validated separately. Floating-point comparisons
used the explicitly defined numerical tolerances from
the corresponding audit steps.
</p>

<h2>4. Injection and Scheduling</h2>

<p>
14,375 scheduled channel-day additions were independently
recalculated using Formula B:
Injected Count = max(1, ceil(Natural Count × Multiplier)).
</p>

<p>
The reconstructed schedules included 12,636 attack days,
2,374 scheduled Off days, and 54,624 eligible days outside
the attack schedule. The independently validated schedule
contained 4,379 primary-channel switches.
</p>

<h2>5. Historical Baseline Provenance</h2>

{html_table_357(baseline_report_356)}

<p>
All tested baseline values matched independent calculations
using the pre-2011 eligible historical User-Day records.
</p>

<h2>6. Stealth-Level Statistical Diagnostics</h2>

{html_table_357(severity_357)}

<figure>
<img src="{chart_p95_357}" alt="P95 exceedance by stealth">
</figure>

<figure>
<img src="{chart_ratio_357}" alt="Final activity to P95 ratio">
</figure>

<p class="note">
The progression from High to Low is a descriptive,
aggregate-level result. It does not guarantee every
individual user's attack follows a strictly increasing
severity pattern.
</p>

<h2>7. Rolling-Window Completeness</h2>

{html_table_357(windows_357)}

<figure>
<img src="{chart_windows_357}" alt="Rolling-window completeness">
</figure>

<h2>8. Frozen Source Context</h2>

{html_table_357(context_audit_356)}

<p>
Byte-related daily values remained unchanged by count-level
simulation. The scope of this table is limited to the
columns explicitly listed above.
</p>

<h2>9. Temporal Causality and Fault Detection</h2>

<p>
Nine users were independently tested at historical cutoffs.
The resulting {temporal_comparisons_357:,} feature comparisons
matched the saved outputs.
</p>

{html_table_357(fault_report_356)}

<p>
All five tested in-memory corruptions were detected.
No corrupted copies were saved.
</p>

<h2>10. File Integrity and Reproducibility</h2>

<p>
SHA-256 fingerprints identify the source and output
file versions used during the QA process.
An unchanged fingerprint demonstrates byte-level identity
with the version captured during the first validation step.
</p>

{html_table_357(manifest_357)}

<h2>11. Limitations and Remaining Responsibilities</h2>

<ol>{limitations_html_357}</ol>

<h2>12. Final Conclusion</h2>

<p>
<strong>Overall QA Status: PASS.</strong>
The simulated attack quantities, approved schedules,
115-column feature data, historical baseline statistics,
and three Ground Truth tables passed the specified
independent mathematical and consistency checks.
</p>

<p>
The validated datasets are ready for controlled ML
experimentation. Model training, leakage-safe evaluation,
exfiltration-class performance, and external realism
assessment remain separate tasks.
</p>

<footer>
MUBSIR — Stage 3.5 Independent Quality Assurance |
Original datasets and final simulation files unchanged.
</footer>

</body>
</html>
"""

# ------------------------------------------------------------
# 9. Create a NEW report directory.
#    Never overwrite previous reports.
# ------------------------------------------------------------

REPORT_ROOT_357.mkdir(
    parents=True,
    exist_ok=True
)

run_id_357 = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )
    + "_"
    + uuid4().hex[:8]
)

REPORT_DIR_357 = (
    REPORT_ROOT_357
    / f"Stage_3_5_QA_{run_id_357}"
)

REPORT_DIR_357.mkdir(
    parents=False,
    exist_ok=False
)

# ------------------------------------------------------------
# 10. Export report and supporting audit evidence.
# ------------------------------------------------------------

REPORT_FILES_357 = {
    "html": (
        REPORT_DIR_357
        / "MUBSIR_Stage3_5_Independent_QA_Report.html"
    ),
    "checks": (
        REPORT_DIR_357
        / "MUBSIR_Stage3_5_QA_Checks.csv"
    ),
    "manifest": (
        REPORT_DIR_357
        / "MUBSIR_Stage3_5_File_Manifest.csv"
    ),
    "summary": (
        REPORT_DIR_357
        / "MUBSIR_Stage3_5_QA_Summary.json"
    )
}

REPORT_FILES_357["html"].write_text(
    html_357,
    encoding="utf-8"
)

checks_357.to_csv(
    REPORT_FILES_357["checks"],
    index=False
)

manifest_357.to_csv(
    REPORT_FILES_357["manifest"],
    index=False
)

summary_json_357 = {
    "project": "MUBSIR",
    "stage": "3.5",
    "overall_status": "PASS",
    "formula": "B_Natural_Addition",
    "simulated_users": 670,
    "attack_days": 12636,
    "off_days": 2374,
    "injected_channel_days": 14375,
    "injected_event_counts": 42095,
    "original_columns": 85,
    "new_features": 30,
    "final_columns": 115,
    "original_feature_comparisons": original_comparisons_357,
    "new_feature_comparisons": new_comparisons_357,
    "ground_truth_comparisons": gt_comparisons_357,
    "total_compared_values": total_comparisons_357,
    "unexplained_mismatches": 0,
    "historical_baseline_comparisons": historical_comparisons_357,
    "temporal_prefix_comparisons": temporal_comparisons_357,
    "intentional_faults_tested": faults_tested_357,
    "intentional_faults_detected": faults_detected_357,
    "mandatory_checks": len(checks_357),
    "mandatory_checks_passed": int(
        checks_357["Passed"].sum()
    ),
    "generated_utc": generated_utc_357
}

REPORT_FILES_357["summary"].write_text(
    json.dumps(
        summary_json_357,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 11. Verify the exported QA report files.
# ------------------------------------------------------------

saved_report_rows_357 = []

for category, path in REPORT_FILES_357.items():

    assert path.is_file()
    assert path.stat().st_size > 0

    saved_report_rows_357.append({
        "Type": category.upper(),
        "File": path.name,
        "Size_KB": round(
            path.stat().st_size / 1024,
            2
        ),
        "Status": "SAVED"
    })

# Read back the supporting evidence.
reloaded_checks_357 = pd.read_csv(
    REPORT_FILES_357["checks"]
)

reloaded_manifest_357 = pd.read_csv(
    REPORT_FILES_357["manifest"]
)

reloaded_summary_357 = json.loads(
    REPORT_FILES_357["summary"].read_text(
        encoding="utf-8"
    )
)

assert len(reloaded_checks_357) == len(checks_357)

assert len(reloaded_manifest_357) == len(manifest_357)

assert reloaded_checks_357["Passed"].all()

assert reloaded_summary_357[
    "total_compared_values"
] == 8756986

assert reloaded_summary_357[
    "overall_status"
] == "PASS"

print("\nH. EXPORTED QA REPORT FILES")

saved_report_inventory_357 = pd.DataFrame(
    saved_report_rows_357
)

display(saved_report_inventory_357)

# ------------------------------------------------------------
# 12. Final status.
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STAGE 3.5 — FINAL INDEPENDENT QA COMPLETED")
print("=" * 80)

print("Overall status: PASS")
print("Mandatory checks:", len(checks_357))
print("Passed:", int(checks_357["Passed"].sum()))
print("Compared feature/GT values:", total_comparisons_357)
print("Unexplained mismatches: 0")
print("Original and final simulation files: UNCHANGED")
print("QA report export: PASS")
print("Report read-back: PASS")
print("Report directory:", REPORT_DIR_357)
print("STAGE 3.5 STATUS: COMPLETED")


STAGE 3.5 — STEP 3.5.7
FINAL INDEPENDENT QUALITY ASSURANCE REPORT

A. INDEPENDENT VALIDATION COVERAGE


,Validation,Comparisons,Mismatches
0,Original 83 non-key columns,5779622,0
1,30 new behavioral features,2089020,0
2,Three Ground Truth tables,888344,0



B. FINAL MANDATORY QUALITY GATES


,Stage,Check,Passed,Evidence
0,3.5.1,Channel-Day injection reconciliation,True,All saved channel-day quantities match indepen...
1,3.5.1,Daily channel-count reconciliation,True,"69,634 user-days compared for Email, USB and HTTP"
2,3.5.2,Scenario catalog mathematics,True,"1,143 approved scenarios checked"
3,3.5.2,Scheduling and channel validity,True,670 independently checked user schedules
4,3.5.2,Day-type totals,True,"Off, Single, Two and Three-Channel totals"
5,3.5.3,Original feature values,True,"5,779,622 value comparisons"
6,3.5.3,Original feature null placement,True,All 83 non-key columns
7,3.5.3,Aggregate event-count identities,True,"Email, File and HTTP event identities"
8,3.5.4,Thirty new feature values,True,"2,089,020 value comparisons"
9,3.5.4,New feature null placement,True,"5, 10, 20 and 30 WD completeness"



C. FILE INTEGRITY MANIFEST


,Dataset,Size_Bytes,Unchanged_Since_3_5_1
0,Original User-Day,24648454,True
1,Original Baseline,928190,True
2,Approved Scenario Catalog,25236,True
3,Validated Schedule,67204,True
4,GT Users,22176,True
5,GT Channel-Days,71935,True
6,GT User-Days,92480,True
7,Final Features,7376590,True
8,Historical Baseline Parquet,221908,None


... (5 outputs left out)

,Window_WD,Complete_Windows,Incomplete_Windows,TVD_Undefined_Complete_Windows,Complete_%
0,5,66954,2680,101,96.15
1,10,63606,6028,4,91.34
2,20,56920,12714,0,81.74
3,30,50241,19393,0,72.15



H. EXPORTED QA REPORT FILES


,Type,File,Size_KB,Status
0,HTML,MUBSIR_Stage3_5_Independent_QA_Report.html,142.23,SAVED
1,CHECKS,MUBSIR_Stage3_5_QA_Checks.csv,1.51,SAVED
2,MANIFEST,MUBSIR_Stage3_5_File_Manifest.csv,2.40,SAVED
3,SUMMARY,MUBSIR_Stage3_5_QA_Summary.json,0.77,SAVED



STAGE 3.5 — FINAL INDEPENDENT QA COMPLETED
Overall status: PASS
Mandatory checks: 19
Passed: 19
Compared feature/GT values: 8756986
Unexplained mismatches: 0
Original and final simulation files: UNCHANGED
QA report export: PASS
Report read-back: PASS
Report directory: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Attack_Simulation\validation_reports\Stage_3_5_QA_20261026T132721Z_9ee5d80e
STAGE 3.5 STATUS: COMPLETED


In [10]:

# Stage 3.5 — Setup: Install plotting dependency
# Installs matplotlib in the current Jupyter kernel environment.

%pip install matplotlib


   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.3 MB 3.1 MB/s eta 0:00:03
   ----- ---------------------------------- 1.3/9.3 MB 2.9 MB/s eta 0:00:03
   -------- ------------------------------- 2.1/9.3 MB 3.6 MB/s eta 0:00:03
   -------------- ------------------------- 3.4/9.3 MB 4.0 MB/s eta 0:00:02
   -------------------- ------------------- 4.7/9.3 MB 4.4 MB/s eta 0:00:02
   ------------------------- -------------- 6.0/9.3 MB 4.7 MB/s eta 0:00:01
   ------------------------------- -------- 7.3/9.3 MB 5.0 MB/s eta 0:00:01
   -------------------------------------- - 8.9/9.3 MB 5.3 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 5.2 MB/s  0:00:01
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ------------------------- -------------- 1.6/2.5 MB 8.7 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 7.3 MB/s  0:00:00
   -----------------------------


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:

import matplotlib
import matplotlib.pyplot as plt

print("Matplotlib version:", matplotlib.__version__)
print("Matplotlib import: PASS")


Matplotlib is building the font cache; this may take a moment.


Matplotlib version: 3.11.2
Matplotlib import: PASS
